In [1]:
import os
from copy import deepcopy
import json
# import xml.etree.ElementTree as ET
# from rich.tree import Tree
# from rich import print as rprint
import io
from typing import List, Union, Tuple, Dict, Optional
from collections.abc import Iterable
from tqdm import tqdm
# import pdfplumber
# import fitz 
import numpy as np
import pandas as pd
import requests
# import xmltodict
import re
from pypdf import PdfReader, PdfWriter
from difflib import SequenceMatcher
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.document import DocumentStream
from docling.pipeline.vlm_pipeline import VlmPipeline
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
# device = torch.device("mps")
from table_link_to_excel import _curl_get_text, _extract_pmc_info, _try_pmc_direct_table_download, _flatten_columns, sanitize_sheet_name, fetch_html, fetch_pmc_fulltext_xml, pick_table, table_to_dataframe, _clean_text
from bs4 import BeautifulSoup
from gwas_formatting_engine import GWASFormattingEngine
from gwas_information_retriever import GWASInformationRetriever
from gwas_table_extraction import *

/Users/justpqa/advpai/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Try to look at the pdf

Try to look at the website of the paper instead

Try to use Docling

In [2]:
# # first we need to know the number of col of a table to make sure we try the right orientation with docling
# def extract_tables_num_col_lst_from_pmc(pmcid):
#     url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"

#     response = requests.get(url)
#     tables_num_col_lst = []
#     if response.status_code == 200:
#         data = response.json()
#         for d in data:
#             doc = d["documents"]
#             for p in doc:
#                 passage = p["passages"]
#                 for item in passage:
#                     if item.get("infons", "").get("type", "").lower() == "table" and "text" in item:
#                         table_str = item["text"]
#                         num_col = 0
#                         for row in table_str.split("\t \t"):
#                             row_lst = row.split("\t")
#                             num_col = max(num_col, len(row_lst))
#                         tables_num_col_lst.append(num_col)
#         print(f"Successfully retrieve number of columns")
#     else:
#         print(f"Failed to retrieve number of columns: {response.status_code}")
        
#     return tables_num_col_lst

In [3]:
# # Cleaning value with tag
# def clean_cell(val):
#     tag_pattern = r'\s[a-z]$'
#     if isinstance(val, str):
#         return re.sub(tag_pattern, '', val)
#     return val

# def clean_headers(df):
#     tag_pattern = r'\s[a-z]$'
#     new_cols = []
#     seen = {}
#     for col in df.columns:
#         if pd.isna(col):
#             new_cols.append("")
#         else:
#             # 1. Apply the regex to the name string
#             clean_name = re.sub(tag_pattern, '', str(col))
#             # 2. Handle duplicates (e.g., if 'Price a' and 'Price b' both become 'Price')
#             if clean_name in seen:
#                 seen[clean_name] += 1
#                 clean_name = f"{clean_name}_{seen[clean_name]}"
#             else:
#                 seen[clean_name] = 0
#             new_cols.append(clean_name)
#     df.columns = new_cols
#     return df

In [4]:
# # detect specific stuff like snp and pvalue
# def contains_valid_snp(row):
#     row_snp_pattern = r"\b(?:rs|s)\d+\S*"
#     return any(re.search(row_snp_pattern, str(value)) for value in row)

# def contains_valid_pvalue(row):
#     row_pvalue_pattern = r"\d+\.\d+"
#     return any(re.search(row_pvalue_pattern, str(value)) for value in row)

In [5]:
# def extract_tables_lst_from_pdf_and_num_col(file_name: str, tables_num_col_lst: List[int]) -> List[pd.DataFrame]:
#     """
#     Extract tables from a paper given a file_name string and a list of number of col for each tables 
#     (for cross-check to see if we need to do rotation)
#     """
#     reader = PdfReader(file_name)
#     options = PdfPipelineOptions()
#     options.table_structure_options.mode = TableFormerMode.ACCURATE
#     ocr_converter = DocumentConverter(
#         format_options={
#             InputFormat.PDF: PdfFormatOption(pipeline_options=options)
#         }
#     )
#     df_lst = []

#     page_num = 1
#     while len(df_lst) < len(tables_num_col_lst) and page_num <= len(reader.pages):
#         filled = False # flag if we found table
#         for angle in [0, 90]: # Try normal, then try rotated
            
#             writer = PdfWriter()
#             page = reader.pages[page_num - 1]
            
#             if angle != 0:
#                 page.rotate(angle)
            
#             writer.add_page(page)
            
#             # Convert just this one page
#             pdf_buffer = io.BytesIO()
#             writer.write(pdf_buffer)
#             pdf_buffer.seek(0)
            
#             doc_stream = DocumentStream(name=f"page_{page_num}.pdf", stream=pdf_buffer)
#             result = ocr_converter.convert(doc_stream)

#             # Check if this rotation produced valid table rows
#             temp_dfs = []
#             for table in result.document.tables:
#                 df = table.export_to_dataframe()
#                 # check if table is empty
#                 if (not df.empty):
#                     # Case 1: continue from previous table
#                     if len(df_lst) > 0 and df.shape[1] == df_lst[-1].shape[1]:
#                         # extra filters for tables that are snp related, we need to remove rows that do not have snp id
#                         # often are separation between sections
#                         for col in df.columns:
#                             # first modify "" -> nan
#                             df[col] = df[col].replace(r'^\s*$', np.nan, regex=True).ffill()
#                         df["valid_row"] = df.apply(lambda x: contains_valid_snp(x) and contains_valid_pvalue(x), axis=1)
#                         df = df[df["valid_row"]].drop("valid_row", axis=1).reset_index().drop("index", axis = 1)
#                         # some cell have the tag the end (often include a space and a small letter)
#                         df = df.map(clean_cell) 
#                         df = clean_headers(df) 
#                         if df_lst[-1].columns.equals(df.columns):
#                             df_lst[-1] = pd.concat([df_lst[-1], df], ignore_index = True)
#                             filled = True
#                         elif df.shape[1] == tables_num_col_lst[len(df_lst) + len(temp_dfs)]:
#                             # fail that test => add to temp since this is a new table
#                             temp_dfs.append(df)
#                             filled = True
#                     # Case 2: new table
#                     elif (len(df_lst) + len(temp_dfs)) < len(tables_num_col_lst) and df.shape[1] == tables_num_col_lst[len(df_lst) + len(temp_dfs)]: 
#                         # extra filters for tables that are snp related, we need to remove rows that do not have snp id
#                         # often are separation between sections
#                         for col in df.columns:
#                             # first modify "" -> nan
#                             df[col] = df[col].replace(r'^\s*$', np.nan, regex=True).ffill()
#                         df["valid_row"] = df.apply(lambda x: contains_valid_snp(x) and contains_valid_pvalue(x), axis=1)
#                         df = df[df["valid_row"]].drop("valid_row", axis=1).reset_index().drop("index", axis = 1)
#                         # some cell have the tag the end (often include a space and a small letter)
#                         df = df.map(clean_cell) 
#                         df = clean_headers(df)             
#                         temp_dfs.append(df)
#                         filled = True
#             if temp_dfs:
#                 df_lst.extend(temp_dfs)
            
#             if filled:
#                 break
            
#         page_num += 1

#     # final filter, since we only consider df with snp
#     df_lst = [df for df in df_lst if df.shape[0] > 0]
#     return df_lst

In [6]:
# def extract_tables_lst_from_paper(pmcid, file_name, table_inx_to_extract=[]):
#     tables_num_col_lst = extract_tables_num_col_lst_from_pmc(pmcid)
#     if len(table_inx_to_extract) > 0:
#         tables_num_col_lst = [tables_num_col_lst[i] for i in table_inx_to_extract]
#     df_lst = extract_tables_lst_from_pdf_and_num_col(file_name, tables_num_col_lst)
#     return df_lst

In [7]:
# pmcid = "PMC10497850"
# file_name = "papers/ACEL-22-e13938.pdf"
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [8]:
# pmcid = "PMC10115645"
# file_name = "papers/41591_2023_Article_2268.pdf"  # Can be a local path or a URL
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [9]:
# pmcid = "PMC9622429"
# file_name = "papers/nihms-1797266.pdf"  # Can be a local path or a URL
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)  
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [10]:
# pmcid = "PMC10615750"
# file_name = "papers/Recent paper on AD GWAS (1).pdf"  # Can be a local path or a URL
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)  
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [11]:
# pmcid = "PMC6677735"
# file_name = "papers/s42003-019-0537-9.pdf"  # Can be a local path or a URL
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)  
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [12]:
# pmcid = "PMC10286470"
# file_name = "papers/s13024-023-00633-4.pdf"  # Can be a local path or a URL
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)  
# for i, df in enumerate(df_lst):
#     if df.shape[0] > 0:
#         df_lst[i].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)

In [13]:
# pmcid = "PMC9945061"
# file_name = "PMID36809323_table_3.pdf"
# df_lst = extract_tables_lst_from_paper(pmcid, file_name)  
# for i in [3]:
#     df_lst[i-3].to_csv(f"tables/{file_name.split('/')[-1].replace('.pdf', '')}_table_{i}.csv", index=False)
# problem in PMC api

Try to map columns to reference columns

In [14]:
# # Now try to map the columns with the actual col in advp
# referencing_col_df = pd.read_csv("Rules for harmonizing ADVP papers - Main cols.csv")
# referencing_col_lst = referencing_col_df["column"].to_list()
# referencing_col_context_lst = referencing_col_df.apply(lambda x: x["column"] if pd.isna(x["description"]) else x["column"] + ": " + x["description"], axis = 1).to_list()

In [15]:
# # NOTE: before working in the main cell for mapping columns, write inference code here + setting up models here
# def create_embeddings_from_model(sentences, model, tokenizer):
#     input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

#     # get token embeddings
#     with torch.no_grad():
#         output = model(**input)
#     token_embeddings = output[0]

#     # extract mask and mean pooling for sentence embeddings
#     input_mask_expanded = input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
#     sentence_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

#     # final normalization
#     sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

#     return sentence_embeddings

# # setting up models
# embeddings_model_name = "NeuML/pubmedbert-base-embeddings"
# embeddings_model_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_name)
# embeddings_model = AutoModel.from_pretrained(embeddings_model_name)
# embeddings_model.eval()

# referencing_col_embeddings = create_embeddings_from_model(referencing_col_context_lst, embeddings_model, embeddings_model_tokenizer)

In [16]:
# class SingleTokenBiasProcessor(LogitsProcessor):
#     def __init__(self, token_ids, bias_value):
#         self.token_ids = token_ids
#         self.bias_value = bias_value

#     def __call__(self, input_ids, scores):
#         # Create a mask for allowed tokens
#         mask = torch.full_like(scores, -float("inf"))
#         for tid in self.token_ids:
#             mask[:, tid] = self.bias_value
#         return scores + mask

# def reranking_with_llm(col, candidates, llm_model, llm_model_tokenizer):
#     """
#     LLM acts as a re-ranker to pick the best match from a list of candidates.
#     """
#     # Format the candidates as a numbered list for the LLM

#     prompt = f"""Task: Map clinical table headers to GWAS standard ontology, return a single number for the best choice

# Header: "p-value: 0.001, 5e-8, 0.43"
# Candidates: 
#     1. P-value: The statistical significance of the association. Keywords: P, P-value, P_adj, FDR. Examples: 5.0E-08, 0.0012, 1.2 x 10^-5, 0.05.
#     2. Effect Size: The magnitude and direction of the association. Keywords: Beta, OR, HR, Estimate. Examples: Beta=0.25, OR=1.45, HR=1.12, Log(OR)=0.37.
#     3. SNP: Variant identifier, or snp idenifier, or chr:pos. Keywords: chr:position, chr:pos, Variant, rsID, RS number, MarkerName, rs. Examples: rs12345, 20:45269867, 19:45411941:T:C, chr19:45411941, rs429358 (APOE ε4).
# Best Match: 1

# Header: "rs_number: rs123, rs456, rs789"
# Candidates: 
#     1. Chr: Genomic chromosome identifier. Keywords: CHR, Chrom, Chromosome. Examples: 1, 19, X, chr19, chrX.
#     2. Position: Genomic coordinate location. Keywords: BP, POS, Base Pair, start, end. Examples: 45411941, 10240500:10248600 (range), build 37.
#     3. SNP: Variant identifier, or snp idenifier, or chr:pos. Keywords: chr:position, chr:pos, Variant, rsID, RS number, MarkerName, rs. Examples: rs12345, 20:45269867, 19:45411941:T:C, chr19:45411941, rs429358 (APOE ε4).
# Best Match: 2

# Header: "{col}"
# Candidates: 
#     - {candidates[0]}
#     - {candidates[1]}
#     - {candidates[2]}
# Best Match: """

#     allowed_indices = [str(i+1) for i in range(len(candidates))]
#     allowed_token_ids = [llm_model_tokenizer.encode(idx, add_special_tokens=False)[0] for idx in allowed_indices]
    
#     # logit bias to limit tokens that can be output
#     bias_processor = SingleTokenBiasProcessor(allowed_token_ids, 100.0)
#     logits_processor = LogitsProcessorList([bias_processor])

#     # 4. Generate exactly ONE token
#     inputs = llm_model_tokenizer(prompt, return_tensors="pt").to(device)
    
#     with torch.no_grad():
#         output = llm_model.generate(
#             **inputs,
#             max_new_tokens=1,      # Force exactly one token
#             logits_processor=logits_processor, # Force it to be one of our numbers
#             pad_token_id=llm_model_tokenizer.eos_token_id,
#             do_sample=False        # Greedy decoding for consistency
#         )

#     # 5. Extract and Convert to Integer
#     new_token = output[0][-1]
#     predicted_text = llm_model_tokenizer.decode(new_token).strip()
    
#     try:
#         idx = int(predicted_text) - 1 # Convert back to 0-based list index
#         return candidates[idx]
#     except (ValueError, IndexError):
#         return col

# llm_model_name = "stanford-crfm/BioMedLM"
# llm_model_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
# llm_model = AutoModelForCausalLM.from_pretrained(
#     llm_model_name,
#     torch_dtype=torch.bfloat16
# ).to(device)
# llm_model.eval()

In [17]:
# # Try to first convert any abbreviation, then we match column with right semantic
# gwas_abbreviation_dict = {
#     "CHR": "Chromosome number",
#     "BP": "Base-pair position",
#     "POS": "Position",
#     "SNP": "Single nucleotide polymorphism identifier",
#     "RS": "Reference Single nucleotide polymorphism",
#     "VAR": "Variant",
#     "ID": "identifier",
#     "A1": "Effect allele / tested allele",
#     "A2": "Other allele / non-effect allele",
#     "REF": "Reference allele (genome reference)",
#     "ALT": "Alternate allele",
#     "EA": "Effect Allele",
#     "NEA": "Non-Effect Allele",
#     "RA": "Risk Allele",
#     "OA": "Other Allele",
#     "AF": "Allele Frequency (general term)",
#     "RAF": "Risk Allele Frequency",
#     "EAF": "Effect Allele Frequency",
#     "MAF": "Minor Allele Frequency",
#     "BETA": "Effect size (regression coefficient)",
#     "OR": "Odds Ratio",
#     "SE": "Standard Error of effect estimate",
#     "Z": "Z-score statistic",
#     "T": "T-statistic",
#     "CI": "Confidence Interval",
#     "P": "P-value",
#     "PVAL": "P-value",
#     "LOGP": "Negative log10 P-value",
#     "Q": "Heterogeneity statistic (meta-analysis)",
#     "I2": "I-squared heterogeneity metric",
#     "HWE": "Hardy-Weinberg Equilibrium test statistic",
#     "INFO": "Imputation quality score",
#     "R2": "Imputation accuracy metric",
#     "CALLRATE": "Genotype call rate",
#     "MISSING": "Missing genotype rate",
#     "N": "Total sample size",
#     "N_CASES": "Number of cases (for binary traits)",
#     "N_CONTROLS": "Number of controls (for binary traits)",
#     "EUR": "European ancestry",
#     "AFR": "African ancestry",
#     "ASN": "Asian ancestry",
#     "AMR": "Admixed American ancestry",
#     "SAS": "South Asian ancestry",
#     "EAS": "East Asian ancestry",
#     "LD": "Linkage Disequilibrium",
#     "DPRIME": "LD D’ value",
#     "CADD": "CADD score (functional impact)",
#     "EQTL": "Expression quantitative trait locus",
#     "PQTL": "Protein QTL",
#     "GWGAS": "Gene-wide association study",
#     "PRS": "Polygenic Risk Score",
#     "PGS": "Polygenic Score",
#     "QC": "Quality Control",
#     "MA": "Meta-analysis",
#     "HLA": "Human Leukocyte Antigen region",
#     "HR": "Hazard ratio",
#     "HET": "Heterogeneity test",
#     "APOE4": "APOE ε4",
#     "APOE*4": "APOE ε4",
#     "#": "Number of",
#     "frq": "Frequency",
#     'β': "Effect",
#     "nsnps": "Number of Variants"
# }

In [18]:
# def clean_col(col):
#     """
#     Clean a column by replacing any possible abbreviation with their actual meaning for better semantic matching
#     """
#     new_col = col
#     for abb in gwas_abbreviation_dict:
#         if re.search(fr"[^a-zA-Z]{abb.lower()}[^a-zA-Z]", new_col.lower()):
#             new_col = re.sub(fr"([^a-zA-Z]){abb.lower()}([^a-zA-Z])", fr"\1{gwas_abbreviation_dict[abb]}\2", new_col.lower())
#         elif re.search(fr"^{abb.lower()}[^a-zA-Z]", new_col.lower()):
#             new_col = re.sub(fr"^{abb.lower()}([^a-zA-Z])", fr"{gwas_abbreviation_dict[abb]}\1", new_col.lower())
#         elif re.search(fr"[^a-zA-Z]{abb.lower()}$", new_col.lower()):
#             new_col = re.sub(fr"([^a-zA-Z]){abb.lower()}$", fr"\1{gwas_abbreviation_dict[abb]}", new_col.lower())
#         elif re.search(fr"^{abb.lower()}$", new_col.lower()):
#             new_col = re.sub(fr"{abb.lower()}", gwas_abbreviation_dict[abb], new_col.lower())
#     # new_col = new_col.replace(".", " ")
#     return new_col

In [19]:
# def make_col_prompt(col, example_values, num_example_values = 5):
#     """
#     Based on the column title and some possible values of that columns, try to make a prompt
#     """
#     col_prompt = f"{col}: "
#     for i in range(min(num_example_values, len(example_values))):
#         col_prompt += f"{example_values[i]}, "
#     return col_prompt

In [20]:
# def match_single_col_to_ref_col(col, ref_col_lst, ref_col_embeddings, embeddings_model, embeddings_model_tokenizer):
#     """
#     match a single column to reference col given column, the list of referencing col and their embeddings
#     """
#     # calculate column embedding
#     # col_embeddings = embeddings_model.encode(col, normalize_embeddings=True)
#     # multi_index_pattern = r".+\..+"
#     # if re.search(multi_index_pattern, col):
#     #     col = col.replace(".", " ")
#     #     # sub_col_lst = col.split(".")
#     #     # sub_col_embeddings = create_embeddings_from_model(sub_col_lst, embeddings_model, embeddings_model_tokenizer)
#     #     # col_embeddings = torch.mean(sub_col_embeddings, dim = 0)
#     #     col = col.replace(".", " ")
#     #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     # else:
#     #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     col_embeddings = col_embeddings.reshape(-1, 1)

#     # calculate similarity score
#     scores = torch.matmul(ref_col_embeddings, col_embeddings).reshape(-1)

#     # sort similairty score
#     top_k_indices = torch.argsort(scores, descending=True)

#     # verify if we even got good enough similarity
#     best_inx = top_k_indices[0].item()
#     best_score = scores[best_inx].item()
#     # second_best_inx = top_k_indices[1].item()
#     # second_best_score = scores[second_best_inx].item()
#     # need a threshold for score or else, just return col
#     # if best_score < 0.4: 
#     #     return (col, 1)
    
#     # # now do rerank
#     # candidates = [ref_col_lst[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#     # candidates_scores = [scores[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#     # best_ref_col, best_ref_col_scores = reranking_from_model(col, candidates, candidates_scores)
#     # return (best_ref_col, best_ref_col_scores)

#     if best_score >= 0.4:
#         candidates_inx = []
#         for i in range(3):
#             candidates_inx.append((top_k_indices[i], scores[top_k_indices[i]]))
#         return candidates_inx
#     return []

# # def match_many_col_to_ref_col(df, ref_col_df, embeddings_model, embeddings_model_tokenizer):
# #     """
# #     Given a list of column and a dataframe (could be dict later on if that fits better),
# #     try to match each column to the best fitted reference col, 
# #     return a dict of ref col : list of (col, cleaned col prompt, score)
# #     """
# #     # prepare the embeddings for reference col since we reuse them
# #     ref_col_lst = ref_col_df["column"].to_list()
# #     ref_col_context_lst = ref_col_df["column_with_context"].to_list()
# #     ref_col_embeddings = create_embeddings_from_model(ref_col_context_lst, embeddings_model, embeddings_model_tokenizer)

# #     # conduct matching
# #     multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
# #     ref_col_to_col_lst = {}
# #     for col in df.columns:

# #         # extract values needed for prompt
# #         cleaned_col = clean_col(col)
# #         example_values = df[col].unique().tolist()
# #         # prompt: {col}: example, need to delete all : first

# #         if re.search(multi_index_pattern, cleaned_col.strip()):
# #             # try to assess each part and see if which one have highest score
# #             best_ref_col, best_score = None, 0
# #             best_cleaned_sub_col_prompt = None
# #             for sub_col in cleaned_col.split("."):

# #                 # make column prompt for each sub col
# #                 cleaned_sub_col_prompt = make_col_prompt(sub_col, example_values)
                
# #                 # matching and compare
# #                 ref_col, score = match_single_col_to_ref_col(cleaned_sub_col_prompt, ref_col_lst, ref_col_embeddings, embeddings_model, embeddings_model_tokenizer)
# #                 if score > best_score and ref_col != cleaned_sub_col_prompt:
# #                     best_ref_col = ref_col
# #                     best_score = score
# #                     best_cleaned_sub_col_prompt = cleaned_sub_col_prompt
            
# #             # if we have a best one vs not
# #             if best_ref_col is not None:
# #                 if best_ref_col not in ref_col_to_col_lst:
# #                     ref_col_to_col_lst[best_ref_col] = []
# #                 ref_col_to_col_lst[best_ref_col].append((col, best_cleaned_sub_col_prompt, score))
# #             else:
# #                 if col not in ref_col_to_col_lst:
# #                     ref_col_to_col_lst[col] = []
# #                 ref_col_to_col_lst[col].append((col, best_cleaned_sub_col_prompt, 1))
# #         else:
# #             # match a single col with best fit referencing col and return a dict
# #             # make the col prompt
# #             # extra steps to remove .
# #             cleaned_col = cleaned_col.replace(".", "")
# #             cleaned_col_prompt = make_col_prompt(cleaned_col, example_values)

# #             # matching for best col
# #             ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings, embeddings_model, embeddings_model_tokenizer)
# #             # if we still get same col
# #             if ref_col == cleaned_col_prompt: 
# #                 ref_col = col
# #             if ref_col not in ref_col_to_col_lst:
# #                 ref_col_to_col_lst[ref_col] = []
# #             ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))
# #         # ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
# #         # if ref_col == cleaned_col_prompt: 
# #         #     ref_col = col
# #         # if ref_col not in ref_col_to_col_lst:
# #         #     ref_col_to_col_lst[ref_col] = []
# #         # ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

# #     return ref_col_to_col_lst

In [21]:
# def make_prompt(col, candidates):
#     prompt = f"""Header: "{col}"
# Choose the best mapping:
# 1) {candidates[0]}
# 2) {candidates[1]}
# 3) {candidates[2]}
# Answer:"""
    
#     return prompt

In [22]:
# prompt_lst = []
# for file_name in os.listdir("tables"):
#     print(file_name)
#     df = pd.read_csv(f"tables/{file_name}")
#     multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
#     col_to_candidates_lst = {}
#     for col in df.columns:
#         # extract values needed for prompt
#         cleaned_col = clean_col(col)
#         example_values = df[col].unique().tolist()
#         # prompt: {col}: example, need to delete all : first

#         if re.search(multi_index_pattern, cleaned_col.strip()):
#             # try to assess each part and see if which one have highest score
#             candidates = []
#             best_cleaned_sub_col_prompt = None
#             for sub_col in cleaned_col.split("."):

#                 # make column prompt for each sub col
#                 cleaned_sub_col_prompt = make_col_prompt(sub_col, example_values)
                
#                 # matching and compare
#                 ref_candidates_lst = match_single_col_to_ref_col(cleaned_sub_col_prompt, referencing_col_lst, referencing_col_embeddings, embeddings_model, embeddings_model_tokenizer)
#                 candidates.extend(ref_candidates_lst)
#             candidates_inx = [c[0] for c in candidates]
#             candidates = [(referencing_col_context_lst[i], s) for i, s in candidates]
#             candidates = sorted(candidates, key = lambda x: x[1], reverse = True)
#             if len(candidates) >= 3:
#                 candidates = [candidates[i][0] for i in range(3)]
#                 ref_candidates = [referencing_col_lst[candidates_inx[i]] for i in range(3)]
#                 prompt = make_prompt(col, candidates)
#                 prompt_lst.append({"text": prompt})
#                 print(col)
#                 print(ref_candidates)
#         else:
#             # match a single col with best fit referencing col and return a dict
#             # make the col prompt
#             # extra steps to remove .
#             cleaned_col = cleaned_col.replace(".", "")
#             cleaned_col_prompt = make_col_prompt(cleaned_col, example_values)

#             # matching for best col
#             candidates = match_single_col_to_ref_col(cleaned_col_prompt, referencing_col_lst, referencing_col_embeddings, embeddings_model, embeddings_model_tokenizer)
#             candidates_inx = [c[0] for c in candidates]
#             candidates = [(referencing_col_context_lst[i], s) for i, s in candidates]
#             if len(candidates) >= 3:
#                 candidates = [candidates[i][0] for i in range(3)]
#                 ref_candidates = [referencing_col_lst[candidates_inx[i]] for i in range(3)]
#                 prompt = make_prompt(col, candidates)
#                 prompt_lst.append({"text": prompt})
#                 print(col)
#                 print(ref_candidates)
#         # ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#         # if ref_col == cleaned_col_prompt: 
#         #     ref_col = col
#         # if ref_col not in ref_col_to_col_lst:
#         #     ref_col_to_col_lst[ref_col] = []
#         # ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))
#     print()

# # with open("train.jsonl", "w", encoding="utf-8") as f:
# #     for item in prompt_lst:
# #         f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [23]:
# class_lst = [None,  # Annotation (nearby).
# None,  # Gene.
# 0,     # Rs-ID.
# 0,     # Chr.
# 0,     # Position.
# 0,     # A1.
# None,  # A2.
# 0,     # European ancestry meta-analysis.Frq
# 2,     # European ancestry meta-analysis.Effect
# None,  # European ancestry meta-analysis.SE
# 2,     # European ancestry meta-analysis.p
# 1,     # Multi-ethnic meta-analysis.Frq
# 2,     # Multi-ethnic meta-analysis.Effect
# None,  # Multi-ethnic meta-analysis.SE
# 1,     # Multi-ethnic meta-analysis.p
# None,  # Replication sample ( n = 8,789).N
# None,  # Replication sample ( n = 8,789).R 2 (%)
# 2,     # Replication sample ( n = 8,789).Frq
# None,  # Replication sample ( n = 8,789).Effect
# None,  # Replication sample ( n = 8,789).SE
# 0,     # Replication sample ( n = 8,789).p
# 0,     # (s13024-023-00633-4_table_1) Chr.
# 0,     # Position.
# 0,     # Variant.
# 1,     # A1/A2 a (MAF).
# None,  # Individual locus.β G (SE)
# None,  # Individual locus.P G
# None,  # Individual locus.β G × Age (SE)
# None,  # Individual locus.P G × Age
# None,  # Individual locus.P Joint
# 0,     # Pleiotropy.P Placo,G
# 0,     # Pleiotropy.P Placo,G × Age
# 0,     # Pleiotropy.P Placo, Joint
# 0,     # (s13024-023-00633-4_table_0) Chr.
# 0,     # Position.
# 0,     # Variant.
# 1,     # A1/A2 a (MAF).
# 2,     # Genetic effects.β G (SE)
# 0,     # Genetic effects.P G
# None,  # Genetic effects.β G × Age (SE)
# 0,     # Genetic effects.P G × Age
# 0,     # Genetic effects.P Joint
# 1,     # (nihms-1797266_table_4) APOE4 stratification NIA-LOAD
# 0,     # SNP (chr:pos)
# 0,     # SNP (rs)
# None,  # N (AD)
# 0,     # Hazard Ratio
# 0,     # Hazard Ratio 95% CI
# 0,     # P-value
# 1,     # (nihms-1797266_table_5) APOE4 stratification.
# 0,     # SNP (rs).
# None,  # Meta-Analysis results.Model
# 0,     # Meta-Analysis results.Hazard Ratio
# 0,     # Meta-Analysis results.Hazard Ratio 95%CI
# 0,     # Meta-Analysis results.P-value
# 0,     # (41591_2023_Article_2268_table_1) GWASmeta-analysis.SNP
# None,  # GWASmeta-analysis.chr:position
# 0,     # GWASmeta-analysis.EA/OA
# 0,     # i-Share(dichotomous).OR(95% CI)
# 0,     # i-Share(dichotomous).P value
# 0,     # i-Share (continuous).β(SE)
# 0,     # i-Share (continuous).P value
# 0,     # Nagahama(dichotomous).OR(95% CI)
# 0,     # Nagahama(dichotomous).P value
# 0,     # Nagahama (continuous).β(SE)
# 0,     # Nagahama (continuous).P value
# 0,     # (41591_2023_Article_2268_table_0) Region
# 0,     # SNPALL
# 1,     # chr:position
# 0,     # EA/OA
# 0,     # EAF
# None,  # Function
# 0,     # Effect (β)
# None,  # SE
# 0,     # P valueEUR
# 0,     # P valueAll
# 0,     # Het P value
# 0,     # (nihms-1797266_table_2) Chr
# 0,     # Variant
# 0,     # Variant (rs)
# 0,     # Data set
# 0,     # MAF
# 0,     # Hazard ratio
# 0,     # P-value
# 1,     # (nihms-1797266_table_3) APOE4 stratification Amish
# 0,     # SNP (chr:pos)
# 0,     # SNP (rs)
# 0,     # Hazard Ratio
# 0,     # Hazard Ratio 95% CI
# 0,     # P-value
# 0,     # (nihms-1797266_table_1) Genome wide Variant
# 0,     # Variant (rs)
# 0,     # MAF
# 0,     # P value
# None,  # Genes
# 0,     # Variants in LD (+/-500Kb).Variant
# 0,     # Variants in LD (+/-500Kb).Variant (rs)
# None,  # Variants in LD (+/-500Kb).r 2
# 2,     # Variants in LD (+/-500Kb).MAF
# 0,     # Variants in LD (+/-500Kb).P value
# 0,     # (ACEL-22-e13938_table_1) Unnamed: 0
# None,  # Gene-gene combination
# 0,     # OR(95% CI.)
# 0,     # p -value (10,000 permutations).APOE *4 + females
# 0,     # p -value (10,000 permutations).APOE *4 + males
# 0,     # p -value (10,000 permutations).APOE *4 - females
# 0,     # p -value (10,000 permutations).APOE *4 - males
# 0,     # (ACEL-22-e13938_table_0) Variant
# 0,     # Chr.
# None,  # Gene SYMBOL
# 0,     # EA
# 0,     # OR
# 0,     # p
# 0,     # OR_1
# 0,     # p_1
# 0,     # (nihms-1797266_table_0) Location and base pair change (build hg19)
# 0,     # SNP (rs)
# 1,     # MAF
# None,  # Variant type
# 0,     # Hazard Ratio
# 0,     # P-value
# None,  # Overlapping Genes
# None,  # Genes within +/- 500Kb
# None,  # (Recent paper on AD GWAS (1)_table_0) Locus
# 0,     # SNP
# 0,     # Chromosome
# 0,     # Position
# 0,     # Effect allele
# None,  # Reference allele
# 0,     # P, MR-MEGA
# 0,     # P, ancestry heterogeneity
# 0,     # P, random effects
# 0,     # beta, random effects
# None,  # SE, random effects
# None,  # I2
# 1,     # Mean effect allele frequency
# 0,     # Minimum effect allele frequency
# 0,     # Maximum effect allele frequency
# None,  # (Recent paper on AD GWAS (1)_table_1) Locus
# 0,     # SNP
# 0,     # Chromosome
# 0,     # Position
# 0,     # Effect allele
# None,  # Reference allele
# None,  # Nsnps_in_credible_set
# None,  # Posterior probability
# 0,     # P, MR-MEGA
# 0,     # P, ancestry heterogeneity
# None,  # chisq, ancestry (% total)
# None   # I2
# ]

# final_prompt_lst = []
# for i, p in enumerate(prompt_lst):
#     if class_lst[i] is not None:
#         new_p = p
#         new_p["label"] = class_lst[i]
#         final_prompt_lst.append(new_p)
#     else:
#         new_p = p
#         new_p["label"] = 3
#         final_prompt_lst.append(new_p)
# with open("train.jsonl", "w", encoding="utf-8") as f:
#     for item in final_prompt_lst:
#         f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [24]:
# def match_single_col_to_ref_col_with_llm(col, ref_col_lst, ref_col_embeddings):
#     """
#     match a single column to reference col given column, the list of referencing col and their embeddings
#     """
#     calculate column embedding
#     col_embeddings = embeddings_model.encode(col, normalize_embeddings=True)
#     multi_index_pattern = r".+\..+"
#     if re.search(multi_index_pattern, col):
#         col = col.replace(".", " ")
#         # sub_col_lst = col.split(".")
#         # sub_col_embeddings = create_embeddings_from_model(sub_col_lst, embeddings_model, embeddings_model_tokenizer)
#         # col_embeddings = torch.mean(sub_col_embeddings, dim = 0)
#         col = col.replace(".", " ")
#         col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     else:
#         col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#     col_embeddings = col_embeddings.reshape(-1, 1)

#     calculate similarity score
#     scores = torch.matmul(ref_col_embeddings, col_embeddings).reshape(-1)

#     sort similairty score
#     top_k_indices = torch.argsort(scores, descending=True)

#     verify if we even got good enough similarity
#     best_inx = top_k_indices[0].item()
#     best_score = scores[best_inx].item()
#     second_best_inx = top_k_indices[1].item()
#     second_best_score = scores[second_best_inx].item()
#     need a threshold for score or else, just return col
#     if best_score < 0.4: 
#         return (col, 1)
    
#     # now do rerank
#     candidates = [ref_col_lst[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#     candidates_scores = [scores[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#     best_ref_col, best_ref_col_scores = reranking_from_model(col, candidates, candidates_scores)
#     return (best_ref_col, best_ref_col_scores)

#     if best_score < 0.4:
#         return col
#     else:
#         extract top 3 candidates
#         candidates = []
#         for i in range(3):
#             inx = top_k_indices[i]
#             candidates.append(ref_col_lst[inx])
#         best_col = reranking_with_llm(col, candidates, llm_model, llm_model_tokenizer)
#         return best_col

# def match_many_col_to_ref_col_with_llm(df, ref_col_df):
#     """
#     Given a list of column and a dataframe (could be dict later on if that fits better),
#     try to match each column to the best fitted reference col, 
#     return a dict of ref col : list of (col, cleaned col prompt, score)
#     """
#     prepare the embeddings for reference col since we reuse them
#     ref_col_lst = ref_col_df["column"].to_list()
#     ref_col_context_lst = ref_col_df["column_with_context"].to_list()
#     ref_col_embeddings = create_embeddings_from_model(ref_col_context_lst, embeddings_model, embeddings_model_tokenizer)

#     conduct matching
#     multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
#     ref_col_to_col_lst = {}
#     for col in df.columns:

#         extract values needed for prompt
#         cleaned_col = clean_col(col)
#         example_values = df[col].unique().tolist()
#         prompt: {col}: example, need to delete all : first

#         if re.search(multi_index_pattern, cleaned_col.strip()):
#             try to assess each part and see if which one have highest score
#             best_ref_col, best_score = None, 0
#             best_cleaned_sub_col_prompt = None
#             for sub_col in cleaned_col.split("."):

#                 make column prompt for each sub col
#                 cleaned_sub_col_prompt = make_col_prompt(sub_col, example_values)
                
#                 matching and compare
#                 ref_col, score = match_single_col_to_ref_col(cleaned_sub_col_prompt, ref_col_lst, ref_col_embeddings)
#                 if score > best_score and ref_col != cleaned_sub_col_prompt:
#                     best_ref_col = ref_col
#                     best_score = score
#                     best_cleaned_sub_col_prompt = cleaned_sub_col_prompt
            
#             if we have a best one vs not
#             if best_ref_col is not None:
#                 if best_ref_col not in ref_col_to_col_lst:
#                     ref_col_to_col_lst[best_ref_col] = []
#                 ref_col_to_col_lst[best_ref_col].append((col, best_cleaned_sub_col_prompt, score))
#             else:
#                 if col not in ref_col_to_col_lst:
#                     ref_col_to_col_lst[col] = []
#                 ref_col_to_col_lst[col].append((col, best_cleaned_sub_col_prompt, 1))
#         else:
#             match a single col with best fit referencing col and return a dict
#             make the col prompt
#             extra steps to remove .
#             cleaned_col = cleaned_col.replace(".", "")
#             cleaned_col_prompt = make_col_prompt(cleaned_col, example_values)

#             matching for best col
#             ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#             if we still get same col
#             if ref_col == cleaned_col_prompt: 
#                 ref_col = col
#             if ref_col not in ref_col_to_col_lst:
#                 ref_col_to_col_lst[ref_col] = []
#             ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))
#         ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#         if ref_col == cleaned_col_prompt: 
#             ref_col = col
#         if ref_col not in ref_col_to_col_lst:
#             ref_col_to_col_lst[ref_col] = []
#         ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

#     return ref_col_to_col_lst

# def match_many_col_to_ref_col(df, ref_col_df, embeddings_model, embeddings_model_tokenizer):
#     """
#     Given a list of column and a dataframe (could be dict later on if that fits better),
#     try to match each column to the best fitted reference col, 
#     return a dict of ref col : list of (col, cleaned col prompt, score)
#     """
#     prepare the embeddings for reference col since we reuse them
#     ref_col_lst = ref_col_df["column"].to_list()
#     ref_col_context_lst = ref_col_df["column_with_context"].to_list()
#     ref_col_embeddings = create_embeddings_from_model(ref_col_context_lst, embeddings_model, embeddings_model_tokenizer)

#     conduct matching
#     multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
#     ref_col_to_col_lst = {}
#     for col in df.columns:

#         extract values needed for prompt
#         cleaned_col = clean_col(col)
#         example_values = df[col].unique().tolist()
#         prompt: {col}: example, need to delete all : first

#         if re.search(multi_index_pattern, cleaned_col.strip()):
#             try to assess each part and see if which one have highest score
#             best_ref_col, best_score = None, 0
#             best_cleaned_sub_col_prompt = None
#             for sub_col in cleaned_col.split("."):

#                 make column prompt for each sub col
#                 cleaned_sub_col_prompt = make_col_prompt(sub_col, example_values)
                
#                 matching and compare
#                 ref_col, score = match_single_col_to_ref_col(cleaned_sub_col_prompt, ref_col_lst, ref_col_embeddings, embeddings_model, embeddings_model_tokenizer)
#                 if score > best_score and ref_col != cleaned_sub_col_prompt:
#                     best_ref_col = ref_col
#                     best_score = score
#                     best_cleaned_sub_col_prompt = cleaned_sub_col_prompt
            
#             if we have a best one vs not
#             if best_ref_col is not None:
#                 if best_ref_col not in ref_col_to_col_lst:
#                     ref_col_to_col_lst[best_ref_col] = []
#                 ref_col_to_col_lst[best_ref_col].append((col, best_cleaned_sub_col_prompt, score))
#             else:
#                 if col not in ref_col_to_col_lst:
#                     ref_col_to_col_lst[col] = []
#                 ref_col_to_col_lst[col].append((col, best_cleaned_sub_col_prompt, 1))
#         else:
#             match a single col with best fit referencing col and return a dict
#             make the col prompt
#             extra steps to remove .
#             cleaned_col = cleaned_col.replace(".", "")
#             cleaned_col_prompt = make_col_prompt(cleaned_col, example_values)

#             matching for best col
#             ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings, embeddings_model, embeddings_model_tokenizer)
#             if we still get same col
#             if ref_col == cleaned_col_prompt: 
#                 ref_col = col
#             if ref_col not in ref_col_to_col_lst:
#                 ref_col_to_col_lst[ref_col] = []
#             ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))
#         ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#         if ref_col == cleaned_col_prompt: 
#             ref_col = col
#         if ref_col not in ref_col_to_col_lst:
#             ref_col_to_col_lst[ref_col] = []
#         ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

#     return ref_col_to_col_lst

In [25]:
# for file in os.listdir("./tables"):
#     if "table" in file and ".csv" in file and "harmonized" not in file:
#         print(file)
#         df = pd.read_csv(f"./tables/{file}")
#         df.columns = ['' if 'Unnamed:' in col else col for col in df.columns]
#         col_to_ref_col = match_many_col_to_ref_col(df, referencing_cols)
#         for ref_col in col_to_ref_col:
#             print(f"{ref_col}: {col_to_ref_col[ref_col]}")
#         print()

Rewrite the matching engine as an object

In [26]:
# class SingleTokenBiasProcessor(LogitsProcessor):
#     def __init__(self, token_ids, bias_value):
#         self.token_ids = token_ids
#         self.bias_value = bias_value

#     def __call__(self, input_ids, scores):
#         # Create a mask for allowed tokens
#         mask = torch.full_like(scores, -float("inf"))
#         for tid in self.token_ids:
#             mask[:, tid] = self.bias_value
#         return scores + mask

In [27]:
# class GWASColumnMatchingEngine:
#     def __init__(self, referencing_col_df: pd.DataFrame, embeddings_model_name: str = "NeuML/pubmedbert-base-embeddings", 
#                  use_llm: bool = False, llm_model_name: str = "stanford-crfm/BioMedLM", device: str = "cpu"):
#         # df of referencing col
#         if not ("column" in referencing_col_df.columns and "description" in referencing_col_df.columns):
#             raise Exception("Error: Dataframe for referencing columns need to have 2 columns: column and description")
#         self.referencing_col_lst = referencing_col_df["column"].to_list()
#         self.referencing_col_context_lst = referencing_col_df.apply(lambda x: x["column"] if pd.isna(x["description"]) else x["column"] + ": " + x["description"], axis = 1).to_list()
#         if use_llm:
#             self.referencing_col_to_col_context = {c: cc for c, cc in zip(self.referencing_col_lst, self.referencing_col_context_lst)}
#             self.referencing_col_context_to_col = {cc: c for c, cc in zip(self.referencing_col_lst, self.referencing_col_context_lst)}

#         # Try to first convert any abbreviation, then we match column with right semantic
#         self.gwas_abbreviation_dict = {
#             "CHR": "Chromosome number",
#             "BP": "Base-pair position",
#             "POS": "Position",
#             "SNP": "Single nucleotide polymorphism identifier",
#             "RS": "Reference Single nucleotide polymorphism",
#             "VAR": "Variant",
#             "ID": "identifier",
#             "A1": "Effect allele / tested allele",
#             "A2": "Other allele / non-effect allele",
#             "REF": "Reference allele (genome reference)",
#             "ALT": "Alternate allele",
#             "EA": "Effect Allele",
#             "NEA": "Non-Effect Allele",
#             "RA": "Risk Allele",
#             "OA": "Other Allele",
#             "AF": "Allele Frequency (general term)",
#             "RAF": "Risk Allele Frequency",
#             "EAF": "Effect Allele Frequency",
#             "MAF": "Minor Allele Frequency",
#             "BETA": "Effect size (regression coefficient)",
#             "OR": "Odds Ratio",
#             "SE": "Standard Error of effect estimate",
#             "Z": "Z-score statistic",
#             "T": "T-statistic",
#             "CI": "Confidence Interval",
#             "P": "P-value",
#             "PVAL": "P-value",
#             "LOGP": "Negative log10 P-value",
#             "Q": "Heterogeneity statistic (meta-analysis)",
#             "I2": "I-squared heterogeneity metric",
#             "HWE": "Hardy-Weinberg Equilibrium test statistic",
#             "INFO": "Imputation quality score",
#             "R2": "Imputation accuracy metric",
#             "CALLRATE": "Genotype call rate",
#             "MISSING": "Missing genotype rate",
#             "N": "Total sample size",
#             "N_CASES": "Number of cases (for binary traits)",
#             "N_CONTROLS": "Number of controls (for binary traits)",
#             "EUR": "European ancestry",
#             "AFR": "African ancestry",
#             "ASN": "Asian ancestry",
#             "AMR": "Admixed American ancestry",
#             "SAS": "South Asian ancestry",
#             "EAS": "East Asian ancestry",
#             "LD": "Linkage Disequilibrium",
#             "DPRIME": "LD D’ value",
#             "CADD": "CADD score (functional impact)",
#             "EQTL": "Expression quantitative trait locus",
#             "PQTL": "Protein QTL",
#             "GWGAS": "Gene-wide association study",
#             "PRS": "Polygenic Risk Score",
#             "PGS": "Polygenic Score",
#             "QC": "Quality Control",
#             "MA": "Meta-analysis",
#             "HLA": "Human Leukocyte Antigen region",
#             "HR": "Hazard ratio",
#             "HET": "Heterogeneity test",
#             "APOE4": "APOE ε4",
#             "APOE*4": "APOE ε4",
#             "#": "Number of",
#             "frq": "Frequency",
#             'β': "Effect",
#             "nsnps": "Number of Variants"
#         }

#         # embeddings model
#         self.embeddings_model_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_name)
#         self.embeddings_model = AutoModel.from_pretrained(embeddings_model_name)
#         self.embeddings_model.eval()

#         # also make col embeddings
#         self.referencing_col_embeddings = self.create_col_embeddings_from_model(self.referencing_col_context_lst)

#         # llm model
#         self.use_llm = use_llm
#         if use_llm:
#             self.llm_model_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
#             self.llm_model = AutoModelForCausalLM.from_pretrained(
#                 llm_model_name,
#                 dtype=torch.bfloat16
#             ).to(device)
#             self.llm_model.eval()

#         # device
#         self.device = device

#     def clean_col(self, col: str) -> str:
#         """
#         Clean a column by replacing any possible abbreviation with their actual meaning for better semantic matching
#         """
#         new_col = col
#         for abb in self.gwas_abbreviation_dict:
#             if re.search(fr"[^a-zA-Z]{abb.lower()}[^a-zA-Z]", new_col.lower()):
#                 new_col = re.sub(fr"([^a-zA-Z]){abb.lower()}([^a-zA-Z])", fr"\1{self.gwas_abbreviation_dict[abb]}\2", new_col.lower())
#             elif re.search(fr"^{abb.lower()}[^a-zA-Z]", new_col.lower()):
#                 new_col = re.sub(fr"^{abb.lower()}([^a-zA-Z])", fr"{self.gwas_abbreviation_dict[abb]}\1", new_col.lower())
#             elif re.search(fr"[^a-zA-Z]{abb.lower()}$", new_col.lower()):
#                 new_col = re.sub(fr"([^a-zA-Z]){abb.lower()}$", fr"\1{self.gwas_abbreviation_dict[abb]}", new_col.lower())
#             elif re.search(fr"^{abb.lower()}$", new_col.lower()):
#                 new_col = re.sub(fr"{abb.lower()}", self.gwas_abbreviation_dict[abb], new_col.lower())
#         # new_col = new_col.replace(".", " ")
#         return new_col
    
#     def make_col_prompt(self, col: str, example_values: Iterable, num_example_values: int = 5) -> str:
#         """
#         Based on the column title and some possible values of that columns, try to make a prompt
#         """
#         col_prompt = f"{col}: "
#         for i in range(min(num_example_values, len(example_values))):
#             col_prompt += f"{example_values[i]}, "
#         return col_prompt

#     def create_col_embeddings_from_model(self, col: str | List[str]) -> np.ndarray | torch.Tensor:
#         """
#         Create embeddings from string represent col name or a prompt of that col
#         """
#         input = self.embeddings_model_tokenizer(col, padding=True, truncation=True, return_tensors='pt')

#         # get token embeddings
#         with torch.no_grad():
#             output = self.embeddings_model(**input)
#         token_embeddings = output[0]

#         # extract mask and mean pooling for sentence embeddings
#         input_mask_expanded = input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
#         col_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

#         # final normalization
#         col_embeddings = F.normalize(col_embeddings, p=2, dim=1)

#         return col_embeddings
    
#     def reranking_with_llm(self, col: str, candidates: List[str]) -> str:
#         """
#         LLM acts as a re-ranker to pick the best match from a list of candidates.
#         """
#         if not self.use_llm:
#             raise Exception("LLM use has not been enabled in the model, please re-init with use_llm=True")
        
#         # Format the candidates as a numbered list for the LLM
#         candidates_str = "\n".join([f"\t{inx+1}. {c}" for inx, c in enumerate(candidates)])
#         prompt = f"""Task: Map clinical table headers to GWAS standard ontology, return a single number for the best choice

# Header: "p-value: 0.001, 5e-8, 0.43"
# Candidates: 
#     1. P-value: The statistical significance of the association. Keywords: P, P-value, P_adj, FDR. Examples: 5.0E-08, 0.0012, 1.2 x 10^-5, 0.05.
#     2. Effect Size: The magnitude and direction of the association. Keywords: Beta, OR, HR, Estimate. Examples: Beta=0.25, OR=1.45, HR=1.12, Log(OR)=0.37.
#     3. SNP: Variant identifier, or snp idenifier, or chr:pos. Keywords: chr:position, chr:pos, Variant, rsID, RS number, MarkerName, rs. Examples: rs12345, 20:45269867, 19:45411941:T:C, chr19:45411941, rs429358 (APOE ε4).
# Best Match: 1

# Header: "rs_number: rs123, rs456, rs789"
# Candidates: 
#     1. Chr: Genomic chromosome identifier. Keywords: CHR, Chrom, Chromosome. Examples: 1, 19, X, chr19, chrX.
#     2. Position: Genomic coordinate location. Keywords: BP, POS, Base Pair, start, end. Examples: 45411941, 10240500:10248600 (range), build 37.
#     3. SNP: Variant identifier, or snp idenifier, or chr:pos. Keywords: chr:position, chr:pos, Variant, rsID, RS number, MarkerName, rs. Examples: rs12345, 20:45269867, 19:45411941:T:C, chr19:45411941, rs429358 (APOE ε4).
# Best Match: 2

# Header: "{col}"
# Candidates: 
# {candidates_str}
# Best Match: """

#         allowed_indices = [str(i+1) for i in range(len(candidates))]
#         allowed_token_ids = [self.llm_model_tokenizer.encode(idx, add_special_tokens=False)[0] for idx in allowed_indices]
        
#         # logit bias to limit tokens that can be output
#         bias_processor = SingleTokenBiasProcessor(allowed_token_ids, 100.0)
#         logits_processor = LogitsProcessorList([bias_processor])

#         # 4. Generate exactly ONE token
#         inputs = self.llm_model_tokenizer(prompt, return_tensors="pt").to(self.device)
        
#         with torch.no_grad():
#             output = self.llm_model.generate(
#                 **inputs,
#                 max_new_tokens=1,      # Force exactly one token
#                 logits_processor=logits_processor, # Force it to be one of our numbers
#                 pad_token_id=self.llm_model_tokenizer.eos_token_id,
#                 do_sample=False        # Greedy decoding for consistency
#             )

#         # 5. Extract and Convert to Integer
#         new_token = output[0][-1]
#         predicted_text = self.llm_model_tokenizer.decode(new_token).strip()
        
#         try:
#             idx = int(predicted_text) - 1 # Convert back to 0-based list index
#             return candidates[idx]
#         except (ValueError, IndexError):
#             return col
        
#     def identify_main_role_in_multi_index_col(self, col: str) -> str:
#         sub_col = col.split(".")
#         candidates = []
#         for i in range(len(sub_col)):
#             main_role = f"{sub_col[i]} of "
#             context = " and ".join([sub_col[j] for j in range(len(sub_col)) if j != i])
#             main_role_and_context = main_role + context
#             candidates.append(main_role_and_context)
#         # make candidate str
#         candidates_str = "\n".join([f"\t{inx+1}. {c}" for inx, c in enumerate(candidates)])
#         prompt = f"""Task: Figure out which is the real meaning of the column in gwas table, return a number for the best choice

# Candidates: 
#     1. Multi-ethnic meta-analysis of Effect
#     2. Effect of Multi-ethnic meta-analysis
# Best Match: 2

# Candidates: 
#     1. p -value (10,000 permutations) of APOE *4 + females
#     2. APOE *4 + females of p -value (10,000 permutations) 
# Best Match: 1

# Candidates:
#     1. GWASmeta-analysis or chr:position
#     2. chr:position or GWASmeta-analysis
# Best Match: 2

# Candidates: 
# {candidates_str}
# Best Match: """
        
#         allowed_indices = [str(i+1) for i in range(len(candidates))]
#         allowed_token_ids = [self.llm_model_tokenizer.encode(idx, add_special_tokens=False)[0] for idx in allowed_indices]
        
#         # logit bias to limit tokens that can be output
#         bias_processor = SingleTokenBiasProcessor(allowed_token_ids, 100.0)
#         logits_processor = LogitsProcessorList([bias_processor])

#         # 4. Generate exactly ONE token
#         inputs = self.llm_model_tokenizer(prompt, return_tensors="pt").to(self.device)
        
#         with torch.no_grad():
#             output = self.llm_model.generate(
#                 **inputs,
#                 max_new_tokens=1,      # Force exactly one token
#                 logits_processor=logits_processor, # Force it to be one of our numbers
#                 pad_token_id=self.llm_model_tokenizer.eos_token_id,
#                 do_sample=False        # Greedy decoding for consistency
#             )

#         # 5. Extract and Convert to Integer
#         new_token = output[0][-1]
#         predicted_text = self.llm_model_tokenizer.decode(new_token).strip()
        
#         try:
#             idx = int(predicted_text) - 1 # Convert back to 0-based list index
#             return sub_col[idx]
#         except (ValueError, IndexError):
#             return col
        
#     def match_single_col_to_ref_col(self, col: str) -> Tuple[str, float]:
#         """
#         match a single column to reference col given column, the list of referencing col and their embeddings
#         """
#         # calculate column embedding
#         # col_embeddings = embeddings_model.encode(col, normalize_embeddings=True)
#         # multi_index_pattern = r".+\..+"
#         # if re.search(multi_index_pattern, col):
#         #     col = col.replace(".", " ")
#         #     # sub_col_lst = col.split(".")
#         #     # sub_col_embeddings = create_embeddings_from_model(sub_col_lst, embeddings_model, embeddings_model_tokenizer)
#         #     # col_embeddings = torch.mean(sub_col_embeddings, dim = 0)
#         #     col = col.replace(".", " ")
#         #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#         # else:
#         #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#         col_embeddings = self.create_col_embeddings_from_model(col)
#         col_embeddings = col_embeddings.reshape(-1, 1)

#         # calculate similarity score
#         scores = torch.matmul(self.referencing_col_embeddings, col_embeddings).reshape(-1)

#         # sort similairty score
#         top_k_indices = torch.argsort(scores, descending=True)

#         # verify if we even got good enough similarity
#         best_inx = top_k_indices[0].item()
#         best_score = scores[best_inx].item()
#         # second_best_inx = top_k_indices[1].item()
#         # second_best_score = scores[second_best_inx].item()
#         # need a threshold for score or else, just return col
#         # if best_score < 0.4: 
#         #     return (col, 1)
        
#         # # now do rerank
#         # candidates = [ref_col_lst[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#         # candidates_scores = [scores[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#         # best_ref_col, best_ref_col_scores = reranking_from_model(col, candidates, candidates_scores)
#         # return (best_ref_col, best_ref_col_scores)

#         if best_score >= 0.4:
#             return (self.referencing_col_lst[best_inx], best_score)
#         return (col, 1)

#     def match_many_col_to_ref_col(self, df: pd.DataFrame) -> Dict:
#         """
#         Given a list of column and a dataframe (could be dict later on if that fits better),
#         try to match each column to the best fitted reference col, 
#         return a dict of ref col : list of (col, cleaned col prompt, score)
#         """

#         # conduct matching
#         multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
#         ref_col_to_col_lst = {}
#         for col in df.columns:

#             # extract values needed for prompt
#             cleaned_col = self.clean_col(col)
#             example_values = df[col].unique().tolist()
#             # prompt: {col}: example, need to delete all : first

#             if re.search(multi_index_pattern, cleaned_col.strip()):
#                 # try to assess each part and see if which one have highest score
#                 best_ref_col, best_score = None, 0
#                 best_cleaned_sub_col_prompt = None
#                 for sub_col in cleaned_col.split("."):

#                     # make column prompt for each sub col
#                     cleaned_sub_col_prompt = self.make_col_prompt(sub_col, example_values)
                    
#                     # matching and compare
#                     ref_col, score = self.match_single_col_to_ref_col(cleaned_sub_col_prompt)
#                     if score > best_score and ref_col != cleaned_sub_col_prompt:
#                         best_ref_col = ref_col
#                         best_score = score
#                         best_cleaned_sub_col_prompt = cleaned_sub_col_prompt
                
#                 # if we have a best one vs not
#                 if best_ref_col is not None:
#                     if best_ref_col not in ref_col_to_col_lst:
#                         ref_col_to_col_lst[best_ref_col] = []
#                     ref_col_to_col_lst[best_ref_col].append((col, best_cleaned_sub_col_prompt, score))
#                 else:
#                     if col not in ref_col_to_col_lst:
#                         ref_col_to_col_lst[col] = []
#                     ref_col_to_col_lst[col].append((col, best_cleaned_sub_col_prompt, 1))
#             else:
#                 # match a single col with best fit referencing col and return a dict
#                 # make the col prompt
#                 # extra steps to remove .
#                 cleaned_col = cleaned_col.replace(".", "")
#                 cleaned_col_prompt = self.make_col_prompt(cleaned_col, example_values)

#                 # matching for best col
#                 ref_col, score = self.match_single_col_to_ref_col(cleaned_col_prompt)
#                 # if we still get same col
#                 if ref_col == cleaned_col_prompt: 
#                     ref_col = col
#                 if ref_col not in ref_col_to_col_lst:
#                     ref_col_to_col_lst[ref_col] = []
#                 ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))
#             # ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#             # if ref_col == cleaned_col_prompt: 
#             #     ref_col = col
#             # if ref_col not in ref_col_to_col_lst:
#             #     ref_col_to_col_lst[ref_col] = []
#             # ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

#         return ref_col_to_col_lst
    
#     def match_single_col_to_ref_col_with_llm(self, col: str, num_candidates: int = 3) -> str:
#         """
#         match a single column to reference col given column, the list of referencing col and their embeddings
#         """
#         if not self.use_llm:
#             raise Exception("LLM use has not been enabled in the model, please re-init with use_llm=True")
        
#         # calculate column embedding
#         # col_embeddings = embeddings_model.encode(col, normalize_embeddings=True)
#         # multi_index_pattern = r".+\..+"
#         # if re.search(multi_index_pattern, col):
#         #     col = col.replace(".", " ")
#         #     # sub_col_lst = col.split(".")
#         #     # sub_col_embeddings = create_embeddings_from_model(sub_col_lst, embeddings_model, embeddings_model_tokenizer)
#         #     # col_embeddings = torch.mean(sub_col_embeddings, dim = 0)
#         #     col = col.replace(".", " ")
#         #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#         # else:
#         #     col_embeddings = create_embeddings_from_model(col, embeddings_model, embeddings_model_tokenizer)
#         col_embeddings = self.create_col_embeddings_from_model(col)
#         col_embeddings = col_embeddings.reshape(-1, 1)

#         # calculate similarity score
#         scores = torch.matmul(self.referencing_col_embeddings, col_embeddings).reshape(-1)

#         # sort similairty score
#         top_k_indices = torch.argsort(scores, descending=True)

#         # verify if we even got good enough similarity
#         best_inx = top_k_indices[0].item()
#         best_score = scores[best_inx].item()
#         # second_best_inx = top_k_indices[1].item()
#         # second_best_score = scores[second_best_inx].item()
#         # need a threshold for score or else, just return col
#         # if best_score < 0.4: 
#         #     return (col, 1)
        
#         # # now do rerank
#         # candidates = [ref_col_lst[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#         # candidates_scores = [scores[inx] for inx in top_k_indices if scores[inx] >= 0.4]
#         # best_ref_col, best_ref_col_scores = reranking_from_model(col, candidates, candidates_scores)
#         # return (best_ref_col, best_ref_col_scores)

#         if best_score < 0.4:
#             return col
#         else:
#             # extract top 3 candidates
#             candidates = []
#             for i in range(num_candidates):
#                 inx = top_k_indices[i]
#                 # need to add the column with context, not just column
#                 candidates.append(self.referencing_col_context_lst[inx])
#             best_col = self.reranking_with_llm(col, candidates)
#             # after this, map back to normal column
#             if best_col in self.referencing_col_context_to_col:
#                 best_col = self.referencing_col_context_to_col[best_col]
#             return best_col

#     def match_many_col_to_ref_col_with_llm(self, df: pd.DataFrame) -> Dict:
#         """
#         Given a list of column and a dataframe (could be dict later on if that fits better),
#         try to match each column to the best fitted reference col, 
#         return a dict of ref col : list of (col, cleaned col prompt, score)
#         """
#         if not self.use_llm:
#             raise Exception("LLM use has not been enabled in the model, please re-init with use_llm=True")
        
#         # prepare the embeddings for reference col since we reuse them
#         # conduct matching
#         multi_index_pattern = r"^.+\..+$" # need multi index pattern for handling multi index
#         ref_col_to_col_lst = {}
#         for col in df.columns:

#             # extract values needed for prompt
#             cleaned_col = self.clean_col(col)
#             example_values = df[col].unique().tolist()
#             # prompt: {col}: example, need to delete all : first

#             # if re.search(multi_index_pattern, cleaned_col.strip()):
#             #     # try to assess each part and see if which one have highest score
#             #     best_candidates = []
#             #     for sub_col in cleaned_col.split("."):

#             #         # make column prompt for each sub col
#             #         cleaned_sub_col_prompt = self.make_col_prompt(sub_col, example_values)
                    
#             #         # matching and compare
#             #         ref_col = self.match_single_col_to_ref_col_with_llm(cleaned_sub_col_prompt)
#             #         # need to convert because we do the reranking again later, we must convert to col with context
#             #         if ref_col in self.referencing_col_to_col_context:
#             #             best_candidates.append(self.referencing_col_to_col_context[ref_col])
#             #         else:
#             #             best_candidates.append(ref_col)
#             #     # if we have a best one vs not
#             #     if len(best_candidates) > 0:
#             #         cleaned_col = cleaned_col.replace(".", " ")
#             #         cleaned_col_prompt = self.make_col_prompt(cleaned_col, example_values)
#             #         best_ref_col = self.reranking_with_llm(cleaned_col_prompt, best_candidates)

#             #         # after getting best ref col, since it is still col with context, need converting back
#             #         if best_ref_col in self.referencing_col_context_to_col:
#             #             best_ref_col = self.referencing_col_context_to_col[best_ref_col]

#             #         if best_ref_col not in ref_col_to_col_lst:
#             #             ref_col_to_col_lst[best_ref_col] = []
#             #         ref_col_to_col_lst[best_ref_col].append((col, cleaned_col_prompt))
#             #     else:
#             #         if col not in ref_col_to_col_lst:
#             #             ref_col_to_col_lst[col] = []
#             #         ref_col_to_col_lst[col].append((col, None))
#             # else:
#             #     # match a single col with best fit referencing col and return a dict
#             #     # make the col prompt
#             #     # extra steps to remove .
#             #     cleaned_col = cleaned_col.replace(".", "")
#             #     cleaned_col_prompt = self.make_col_prompt(cleaned_col, example_values)

#             #     # matching for best col
#             #     ref_col = self.match_single_col_to_ref_col_with_llm(cleaned_col_prompt)
#             #     # if we still get same col
#             #     if ref_col == cleaned_col_prompt: 
#             #         ref_col = col
#             #     if ref_col not in ref_col_to_col_lst:
#             #         ref_col_to_col_lst[ref_col] = []
#             #     ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt))
#             # ref_col, score = match_single_col_to_ref_col(cleaned_col_prompt, ref_col_lst, ref_col_embeddings)
#             # if ref_col == cleaned_col_prompt: 
#             #     ref_col = col
#             # if ref_col not in ref_col_to_col_lst:
#             #     ref_col_to_col_lst[ref_col] = []
#             # ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

#             if re.search(multi_index_pattern, cleaned_col.strip()):
#                 main_role = self.identify_main_role_in_multi_index_col(cleaned_col)
#                 cleaned_col_prompt = self.make_col_prompt(main_role, example_values)
#             else:
#                 cleaned_col = cleaned_col.replace(".", "")
#                 cleaned_col_prompt = self.make_col_prompt(cleaned_col, example_values)
#             ref_col, score = self.match_single_col_to_ref_col(cleaned_col_prompt)
#             # if we still get same col
#             if ref_col == cleaned_col_prompt: 
#                 ref_col = col
#             if ref_col not in ref_col_to_col_lst:
#                 ref_col_to_col_lst[ref_col] = []
#             ref_col_to_col_lst[ref_col].append((col, cleaned_col_prompt, score))

#         return ref_col_to_col_lst

In [28]:
# gwas_col_matching_engine = GWASColumnMatchingEngine(referencing_cols, use_llm = True, device = device)

# for file in os.listdir("./tables"):
#     if "table" in file and ".csv" in file and "harmonized" not in file:
#         print(file)
#         df = pd.read_csv(f"./tables/{file}")
#         df.columns = ['' if 'Unnamed:' in col else col for col in df.columns]
#         col_to_ref_col = gwas_col_matching_engine.match_many_col_to_ref_col(df)
#         for ref_col in col_to_ref_col:
#             print(f"{ref_col}: {col_to_ref_col[ref_col]}")
#         print()

Final formatting of the tables

In [29]:
# # We now can try to use these dictionary to make a final dataset
# # need a list of columns that we sure that there might be multiple copies of it that we can melt them into many rows
# possible_ref_col_to_melt = ["P-value", "Effect Size", "AF"]
# gwas_col_matching_engine = GWASColumnMatchingEngine(referencing_cols)
# def format_original_table(df, remove_unique_col = False):
#     # map the columns
#     new_col_to_old_col_lst = gwas_col_matching_engine.match_many_col_to_ref_col(df)
#     df_with_ref_col = None 
#     new_col_to_not_melt = [] # list of columns that are stable and not need to be melt
#     new_col_to_old_col_lst_to_melt = {}
#     for new_col in new_col_to_old_col_lst:
#         if len(new_col_to_old_col_lst[new_col]) == 1:
#             if (not remove_unique_col) or (remove_unique_col and new_col_to_old_col_lst[new_col][0][2] != 1):
#                 if df_with_ref_col is None:
#                     df_with_ref_col = df[[new_col_to_old_col_lst[new_col][0][0]]]
#                     df_with_ref_col = df_with_ref_col.rename({new_col_to_old_col_lst[new_col][0][0]: new_col}, axis = 1)
#                 else:
#                     df_with_ref_col[new_col] = df[new_col_to_old_col_lst[new_col][0][0]]
#                 # add these single col to the list of not melt
#                 new_col_to_not_melt.append(new_col)
#         else:
#             if new_col in possible_ref_col_to_melt:
#                 old_col_lst = []
#                 for col, _, _ in new_col_to_old_col_lst[new_col]:
#                     old_col_lst.append(col)
#                     if df_with_ref_col is None:
#                         df_with_ref_col = df[[col]]
#                     else:
#                         df_with_ref_col[col] = df[col]
#                 new_col_to_old_col_lst_to_melt[new_col] = old_col_lst.copy()
#             else:
#                 # make multiple copies with notes
#                 for inx, (col, _, _) in enumerate(new_col_to_old_col_lst[new_col]):
#                     if df_with_ref_col is None:
#                         df_with_ref_col = df[[col]]
#                         df_with_ref_col = df_with_ref_col.rename({col: f"{new_col}_{inx + 1}"}, axis = 1)
#                     else:
#                         df_with_ref_col[f"{new_col}_{inx + 1}"] = df[col]
#                     df_with_ref_col[f"{new_col}_{inx + 1} notes"] = col
#                     # add these cols in group but not need to melt
#                     new_col_to_not_melt.append(f"{new_col}_{inx + 1}")
#                     new_col_to_not_melt.append(f"{new_col}_{inx + 1} notes")
#     # Melting stage
#     if len(new_col_to_old_col_lst_to_melt) > 0:
#         # now melting column in same groups
#         # Instead of keep melting, for each group, we make a new dataset of 
#         # [stable col] + [to be melt col] => melt them as a new df
#         # do this for each gorup and then join together based on stable col
#         df_with_melt_col = None 
#         for new_col in new_col_to_old_col_lst_to_melt:
#             temp_df = deepcopy(df_with_ref_col[new_col_to_not_melt + new_col_to_old_col_lst_to_melt[new_col]])
#             # create a temp row id for stable join
#             temp_df["_row_id"] = np.arange(temp_df.shape[0])
#             temp_df = temp_df.melt(
#                 id_vars = new_col_to_not_melt + ["_row_id"],    
#                 value_vars = new_col_to_old_col_lst_to_melt[new_col], 
#                 var_name = f"{new_col} notes",      
#                 value_name = f"{new_col}"
#             )
#             if df_with_melt_col is None:
#                 df_with_melt_col = deepcopy(temp_df)
#             else:
#                 df_with_melt_col = df_with_melt_col.merge(temp_df, how = "inner", on = ["_row_id"] + new_col_to_not_melt)
#         df_with_melt_col = df_with_melt_col.drop("_row_id", axis = 1)
#         return df_with_melt_col
#     else:
#         return df_with_ref_col

In [30]:
# modified_df_all = None
# for file in os.listdir("./tables"):
#     if ".csv" in file and "table" in file and "harmonized" not in file:
#         df = pd.read_csv(f"./tables/{file}")
#         df.columns = ['' if 'Unnamed:' in col else col for col in df.columns]
#         modified_df = format_original_table(df, remove_unique_col = True)
#         modified_df["file_name"] = file
#         if modified_df_all is None:
#             modified_df_all = modified_df.copy()
#         else:
#             modified_df_all = pd.concat([modified_df_all, modified_df], ignore_index = True)
#         modified_df.to_csv(f"./harmonized_tables/{file.replace('.csv', '')}_harmonized.csv", index = False)
# modified_df_all.to_csv("./harmonized_tables/harmonized_table.csv", index = False)

Testing for table harmonization from table_link_to_excel code

In [31]:
# first we need to extract a set of id from table
def extract_table_id_lst_from_pmc(pmcid):
    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"

    response = requests.get(url)
    tables_id_set = set()
    if response.status_code == 200:
        data = response.json()
        for d in data:
            doc = d["documents"]
            for p in doc:
                passage = p["passages"]
                for item in passage:
                    if "table" in item.get("infons", "").get("type", "").lower() and "id" in item.get("infons", ""):
                        tables_id_set.add(item.get("infons", "").get("id", ""))
                        
        print(f"Successfully retrieve list of tables' id")
    else:
        print(f"Failed to retrieve list of tables' id {response.status_code}")
        
    return [id for id in list(tables_id_set) if id != ""]

In [32]:
def contains_valid_snp(row):
    row_snp_pattern = r"\b(?:rs|s)\d+\S*"
    return any(re.search(row_snp_pattern, str(value)) for value in row)

def contains_valid_pvalue(row):
    row_pvalue_pattern = r"\d+\.\d+"
    return any(re.search(row_pvalue_pattern, str(value)) for value in row)

In [33]:
# NOTE: code sometimes fail to extract tables given we have the right ID, fix it
def table_link_to_excel(pmid, pmcid):
    # extract list of table id directly
    table_id_list = extract_table_id_lst_from_pmc(pmcid) 

    # possible_table_id = ["T", "Tab", "Table", "Tbl", "t", "tab", "table", "tbl"]
    has_error = False
    # for i in range(1, num_tables + 1):
    for table_id in table_id_list:
        table_name = f'tables/{pmid}_{pmcid}_{table_id}_from_pmc.xlsx'
        found_table = False
        # for pti in possible_table_id:
        try:
            if BeautifulSoup is None:
                raise RuntimeError("beautifulsoup4 is not installed. Run: python3 -m pip install beautifulsoup4")

            # pmcid, url_table_id = _extract_pmc_info(f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}")
            # resolved_table_id = table_id

            # Fast-path fallback for PMC direct table assets (often bypasses page-level 403).
            if pmcid and table_id:
                direct_tables = _try_pmc_direct_table_download(pmcid, table_id)
                if direct_tables:
                    with pd.ExcelWriter(table_name, engine="openpyxl") as writer:
                        for i, df in enumerate(direct_tables, start=1):
                            df2 = _flatten_columns(df)
                            df2 = df2.ffill()
                            df2.to_excel(writer, index=False, sheet_name=sanitize_sheet_name("", f"table_{i}"))
                            found_table = True

            if not found_table:
                html = ""
                soup = None
                parse_with_xml = False
                try:
                    html = fetch_html(f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}")
                    soup = BeautifulSoup(html, "lxml")
                except Exception as e:
                    # PMC pages may return 403 to scripts; fallback to Europe PMC XML.
                    if pmcid:
                        xml = fetch_pmc_fulltext_xml(pmcid)
                        soup = BeautifulSoup(xml, "xml")
                        parse_with_xml = True
                    else:
                        raise e

                if table_id:
                    tables = pick_table(soup, table_id=table_id, table_selector=None, table_index=0)
                # elif args.all_tables:
                #     if parse_with_xml:
                #         wraps = soup.find_all("table-wrap")
                #         tables = [w.find("table") for w in wraps if w.find("table") is not None]
                #     else:
                #         tables = soup.select(args.table_selector) if args.table_selector else soup.find_all("table")
                #     if not tables:
                #         raise ValueError("No matched tables found.")
                # else:
                #     tables = pick_table(
                #         soup,
                #         table_id=None,
                #         table_selector=args.table_selector,
                #         table_index=args.table_index,
                #     )
                with pd.ExcelWriter(table_name, engine="openpyxl") as writer:
                    for i, table in enumerate(tables, start=1):
                        df = table_to_dataframe(table)
                        caption = ""
                        cap = table.find("caption")
                        if cap:
                            caption = _clean_text(cap.get_text(" ", strip=True))
                        sheet = sanitize_sheet_name(caption, fallback=f"table_{i}")
                        df2 = _flatten_columns(df)
                        df2 = df2.ffill()
                        df2.to_excel(writer, index=False, header=False, sheet_name=sheet)
                        found_table = True
        except Exception as e:
            print(f"Error in extracting table {table_name} with error {e}")
            has_error = True
            
        # if found_table:
        #     # filter table
        #     df = pd.read_excel(table_name)
        #     df["valid_row"] = df.apply(lambda x: contains_valid_snp(x) and contains_valid_pvalue(x), axis=1)
        #     df = df[df["valid_row"]].drop("valid_row", axis=1).reset_index().drop("index", axis = 1)
        #     if df.shape[0] > 0:
        #         with pd.ExcelWriter(table_name, engine="openpyxl") as writer:
        #             df.to_excel(writer, index=False)
        # else:
        #     # delete file
        #     if table_name in os.listdir("tables"):
        #         os.remove(table_name)
    return has_error

In [34]:
# # Now try to map the columns with the actual col in advp
# # to test "NeuML/pubmedbert-base-embeddings", "michiyasunaga/BioLinkBERT-large"
# referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
# gwas_formatting_engine = GWASFormattingEngine(referencing_col_df, embeddings_model_name = "NeuML/pubmedbert-base-embeddings")

In [35]:
# extra func to convert to float
def safe_float(x) -> Optional[float]:
    try:
        if x is None:
            return None
        if isinstance(x, (int, float)):
            v = float(x)
            return None if pd.isna(v) else v

        s = str(x).strip()
        if not s:
            return None

        s = (
            s.replace("×", "x")
             .replace("−", "-")
             .replace("–", "-")
             .replace("\u2212", "-")
        ).strip()

        # remove commas and NBSPs: "1,234" or "1 234"
        s = re.sub(r"[,\u00A0]", "", s)

        # normalize "5.2*e-8" / "5.2 * E-8" -> "5.2e-8"
        s = re.sub(r"(?i)\*\s*e\s*([+-]?\s*\d+)\b", r"e\1", s)
        s = re.sub(r"\s+", "", s)  # helps with "3 E -8" and "4.2 x 10 ^ -5"

        # parse "4.2x10^-5" / "4.2*10^-5" (also works with X)
        sci = re.fullmatch(r"([+-]?(?:\d+(?:\.\d*)?|\.\d+))(?:x|\*)10\^?([+-]?\d+)", s, flags=re.I)
        if sci:
            base = float(sci.group(1))
            exp = int(sci.group(2))
            v = base * (10 ** exp)
            return None if pd.isna(v) else v

        # handles "3e-8" and "3E-8" natively
        v = float(s)
        return None if pd.isna(v) else v
    except Exception:
        return None
    
def safe_int(x) -> Optional[int]:
    try:
        if x is None:
            return None
        if isinstance(x, (int, float)):
            v = int(x)
            if pd.isna(v):
                return None
            return v
        s = str(x).strip()
        s = s.replace(".", "").replace(",", "").replace(" ", "")
        v = int(s)
        if pd.isna(v):
            return None
        return v
    except Exception:
        return None
    
def safe_round(x, num_digits) -> Optional[float]:
    try:
        return round(x, num_digits)
    except Exception:
        return None
    
def extract_first_number_from_str(s) -> Optional[int | float]:
    try:
        if isinstance(s, (int, float)):
            return s
        else:
            pattern = r"[\+\−]?\d+(?:\.\d+)?"
            match = re.search(pattern, s)
            if match:
                return match.group()
            else:
                return None
    except Exception:
        return None
    
def clean_snp(s) -> str:
    try:
        pattern = r"([rR][sS]\d+)"
        match = re.search(pattern, s)
        if match:
            return match.group()
        else:
            return None
    except Exception:
        return None

In [36]:
embeddings_model = AutoModel.from_pretrained("NeuML/pubmedbert-base-embeddings")
embeddings_model_tokenizer = AutoTokenizer.from_pretrained("NeuML/pubmedbert-base-embeddings")

def make_embeddings(sentences: str | List[str]):
    input = embeddings_model_tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

    # get token embeddings
    with torch.no_grad():
        output = embeddings_model(**input)
    token_embeddings = output[0]

    # extract mask and mean pooling for sentence embeddings
    input_mask_expanded = input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
    embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    # final normalization
    embeddings = F.normalize(embeddings, p=2, dim=1)

    # size (# senetences, # dim)
    return embeddings

def calculate_similarity_scores(sentences_1: str | List[str], sentences_2: str | List[str]):
    embeddings_1, embeddings_2 = make_embeddings(sentences_1), make_embeddings(sentences_2)
    return embeddings_1 @ embeddings_2.T 

In [37]:
def combine_possible_info(lst: List[str]):
    return " + ".join([x for x in list(set(lst)) if len(x) > 0])

def combine_possible_info_multilist(multilst: List[List[str]]):
    final_lst = []
    for lst in multilst:
        final_lst.extend(lst)
    return " + ".join([x for x in list(set(final_lst)) if len(x) > 0])

def match_possible_info_to_df(df: pd.DataFrame, col_to_possible_info: Dict, threshold: float = 0.6):
    notes_col = [col for col in df.columns if "notes" in col]
    if len(notes_col) == 0:
        for col in col_to_possible_info:
            if len(col_to_possible_info[col]) == 0:
                df[col] = pd.NA
            else:
                df[col] = combine_possible_info(col_to_possible_info[col])
    else:
        # use info in notes to map, for each info, find the best one
        for col in col_to_possible_info:
            if len(col_to_possible_info[col]) == 0:
                df[col] = pd.NA
            else:
                used_col = []
                for n_col in notes_col:
                    # for each note, check if the match is actually related to that note by check the max similarity
                    unique_value = df[[n_col]].dropna()[n_col].unique().tolist()
                    similarity_score = calculate_similarity_scores(col_to_possible_info[col], unique_value) # #possible info * #unique value
                    # if torch.max(similarity_score) < 0.6:
                    # if best match do not have sim score at least 0.4 - 0.6
                    unique_value_to_possible_info = {}
                    if torch.min(torch.max(similarity_score, dim = 0).values) <= threshold:
                        for i, u in enumerate(unique_value):
                            unique_value_to_possible_info[u] = combine_possible_info(col_to_possible_info[col])
                    else:
                        # best_inx = torch.argmax(similarity_score, dim = 0)
                        # unique_value_to_possible_info = {}
                        # for i, u in enumerate(unique_value):
                        #     unique_value_to_possible_info[u] = col_to_possible_info[col][best_inx[i]]
                        for i, u in enumerate(unique_value):
                            valid_info = [col_to_possible_info[col][inx] for inx in range(similarity_score.shape[0]) if similarity_score[inx, i] >= threshold]
                            unique_value_to_possible_info[u] = combine_possible_info(valid_info)
                    df[f"{col} from {n_col}"] = df[n_col].apply(lambda x: unique_value_to_possible_info.get(x, ""))
                    used_col.append(f"{col} from {n_col}")
                df[col] = df[used_col].apply(lambda x: combine_possible_info(x), axis = 1)
                df[col] = df[col].apply(lambda x: pd.NA if len(x) == 0 else x)
                df = df.drop(used_col, axis = 1)
    return df

def match_possible_info_to_df(df: pd.DataFrame, col_to_possible_info: Dict, threshold: float = 0.4):
    notes_col = [col for col in df.columns if "notes" in col]
    if len(notes_col) == 0:
        for col in col_to_possible_info:
            if len(col_to_possible_info[col]) == 0:
                df[col] = pd.NA
            else:
                df[col] = combine_possible_info(col_to_possible_info[col])
    else:
        # use info in notes to map, for each info, find the best one
        for col in col_to_possible_info:
            if len(col_to_possible_info[col]) == 0:
                df[col] = pd.NA
            else:
                used_col = []
                for n_col in notes_col:
                    # for each note, check if the match is actually related to that note by check the max similarity
                    unique_value = df[[n_col]].dropna()[n_col].unique().tolist()
                    similarity_score = calculate_similarity_scores(col_to_possible_info[col], unique_value) # #possible info * #unique value
                    # if torch.max(similarity_score) < 0.6:
                    # if best match do not have sim score at least 0.4 - 0.6
                    unique_value_to_possible_info = {}
                    if torch.min(torch.max(similarity_score, dim = 0).values) <= threshold:
                        for i, u in enumerate(unique_value):
                            unique_value_to_possible_info[u] = combine_possible_info(col_to_possible_info[col])
                    else:
                        # best_inx = torch.argmax(similarity_score, dim = 0)
                        # unique_value_to_possible_info = {}
                        # for i, u in enumerate(unique_value):
                        #     unique_value_to_possible_info[u] = col_to_possible_info[col][best_inx[i]]
                        for i, u in enumerate(unique_value):
                            valid_info = [col_to_possible_info[col][inx] for inx in range(similarity_score.shape[0]) if similarity_score[inx, i] >= threshold]
                            unique_value_to_possible_info[u] = combine_possible_info(valid_info)
                    df[f"{col} from {n_col}"] = df[n_col].apply(lambda x: unique_value_to_possible_info.get(x, ""))
                    used_col.append(f"{col} from {n_col}")
                df[col] = df[used_col].apply(lambda x: combine_possible_info(x), axis = 1)
                df[col] = df[col].apply(lambda x: pd.NA if len(x) == 0 else x)
                df = df.drop(used_col, axis = 1)
    return df

def match_possible_info_to_df_with_clues(df: pd.DataFrame, pmid: int, pmcid: str, gwas_information_retriever: GWASInformationRetriever):
    notes_col = [col for col in df.columns if "notes" in col]
    if len(notes_col) == 0:
        col_to_possible_info = gwas_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
        for col in col_to_possible_info:
            if len(col_to_possible_info[col]) == 0:
                df[col] = pd.NA
            else:
                df[col] = combine_possible_info(col_to_possible_info[col])
    else:
        # use info in notes to map, for each info, find the best one
        col_to_used_col = {}
        for n_col in notes_col:
            clues = df[[n_col]].dropna()[n_col].unique().tolist()
            col_to_clues_to_possible_info = gwas_information_retriever.extract_possible_info_from_paper_and_clues(pmid, pmcid, clues)
            for col in col_to_clues_to_possible_info:
                if col not in col_to_used_col:
                    col_to_used_col[col] = []
                df[f"{col} from {n_col}"] = df[n_col].apply(lambda x: col_to_clues_to_possible_info[col].get(x, []))
                col_to_used_col[col].append(f"{col} from {n_col}")
        for col in col_to_clues_to_possible_info:
            df[col] = df[col_to_used_col[col]].apply(lambda x: combine_possible_info_multilist(x), axis = 1)
            df[col] = df[col].apply(lambda x: pd.NA if len(x) == 0 else x)
            df = df.drop(col_to_used_col[col], axis = 1)
        return df

In [38]:
test_papers_info = [
    (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
    (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
    (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
    (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
    (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
]
test_papers_info_sample = deepcopy(test_papers_info)
np.random.seed(37)
np.random.shuffle(test_papers_info_sample)
test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30448613, "PMC6331247"), 
# ]

referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
gwas_formatting_engine = GWASFormattingEngine(referencing_col_df)

referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
gwas_information_retriever = GWASInformationRetriever(referencing_col_require_rag_df, use_hf = False, device = "mps")
# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    found_table = False
    try:
        has_error = table_link_to_excel(pmid, pmcid)
        if has_error:
            print(f"Error in extracting from {pmid}-{pmcid} with table_link_to_excel")
        else:
            found_table = True
            print(f"Success in extracting from {pmid}-{pmcid} with table_link_to_excel")
    except Exception as e:
        print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    if not found_table:
        try: 
            df_lst = extract_tables_lst_from_paper(pmcid, f"test_papers/{pmid}_{pmcid}.pdf")
            for i, df in enumerate(df_lst):
                if df.shape[0] > 0:
                    df.to_csv(f"tables/{pmid}_{pmcid}_{i}_from_pdf.csv", index = False)
                    found_table = True
            if found_table:
                print(f"Success in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
            else:
                print(f"Error in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
        except Exception as e:
            print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
        for file_name in os.listdir("tables"):
            if str(pmid) in file_name and pmcid in file_name:
                if ".xlsx" in file_name:
                    df = pd.read_excel(f"tables/{file_name}")
                else:
                    df = pd.read_csv(f"tables/{file_name}")

                # save matching dict for debug
                file_name_to_matching[file_name] = gwas_formatting_engine.match_many_col_to_ref_col(df)

                harmonized_df = gwas_formatting_engine.format_original_table(df, remove_unique_col = True)
                if harmonized_df_all is None:
                    harmonized_df_all = harmonized_df.copy()
                else:
                    harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in harmonizing from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in harmonizing from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
        int_col = ["Chr"]
        for c in int_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
        numerical_col_with_many_numbers = ["Effect"]
        for c in numerical_col_with_many_numbers:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
        float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
        for c in float_col:
            harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
            if float_col[c] is not None:
                harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in converting to number from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in converting to number from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        col_require_rag_to_possible_info = gwas_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
        print(pmid, pmcid)
        print(col_require_rag_to_possible_info)
        harmonized_df_all = match_possible_info_to_df(harmonized_df_all, col_require_rag_to_possible_info)
        # harmonized_df_all = match_possible_info_to_df_with_clues(harmonized_df_all, pmid, pmcid, gwas_information_retriever)
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        # if (pmid, pmcid) in test_papers_info_sample:
        #     harmonized_df_all.to_csv(f"pred_tables_v2/{pmid}_{pmcid}_pred.csv", index = False)
        harmonized_df_all.to_csv(f"pred_tables_v2/{pmid}_{pmcid}_pred.csv", index = False)
        print(f"Success in extracting columns from paper text from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in extracting columns from paper text from {pmid}-{pmcid} with error {e}")

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
llama_context: n_ctx_per_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported

Successfully retrieve list of tables' id
Success in extracting from 30448613-PMC6331247 with table_link_to_excel


/Users/justpqa/advpai/gwas_formatting_engine.py:649: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_ref_col[col] = df[col]
/Users/justpqa/advpai/gwas_formatting_engine.py:649: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_ref_col[col] = df[col]
/Users/justpqa/advpai/gwas_formatting_engine.py:649: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pand

Success in harmonizing from 30448613-PMC6331247
Success in converting to number from 30448613-PMC6331247


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


...

Success in harmonizing from 30805717-PMC7193309
Success in converting to number from 30805717-PMC7193309
30805717 PMC7193309
{'Population': [], 'Cohort': [], 'Stage': ['GLOBAL: cross-trait meta-analysis'], 'Imputation': ['GLOBAL: imputed to 1000 genome', '1000 genome', 'hapmap 2'], 'Study type': ['GLOBAL: GWAS', 'AD', 'IGAP', 'Giant', 'DIAGRAM', ' MAGIC', 'UK Biobank'], 'Phenotype': ['Alzheimer’s disease', 'BMI', 'LDL cholesterol', 'Type 2 diabetes', 'Blood pressure', 'AD', 'hdl', 'whr', 't2d', 'fg', 'fins', 'fg-fins meta-analysis', 'hdl', 'tc', 'tg', 'Alzheimer disease', 'vascular dementia', 'fasting glucose', 'insulin levels', 'diabetes', 'hpa axis', 'diabetes', 'mild cognitive impairment', 'infarcts', 'amyloid', 'tau accumulation']}
Success in extracting columns from paper text from 30805717-PMC7193309
Successfully retrieve list of tables' id
Success in extracting from 30636644-PMC6330399 with table_link_to_excel
Success in harmonizing from 30636644-PMC6330399
Success in converting 

/var/folders/5b/jn4p_4vx7j9gx2jdrmbjlxnr0000gn/T/ipykernel_18839/1058420856.py:71: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)


Success in harmonizing from 28560309-PMC5440281
Success in converting to number from 28560309-PMC5440281
28560309 PMC5440281
{'Population': ['ADNI-1', 'ADNI-2', 'ADNI-GO'], 'Cohort': ['ADNI', 'ADNI-1', 'ADNI-2', 'ADNI-GO'], 'Stage': ['replication', 'GLOBAL: discovery'], 'Imputation': ['GLOBAL: imputed to 1000 Genomes', 'ncbi 1000 genomes build 37 ucsc hg19'], 'Study type': ['WGS', 'Genotyping', 'ADNI-1', 'ADNI-2', 'GLOBAL: genotype imputation'], 'Phenotype': ['AD', 'Alzheimer’s disease', 'AD-related diseases', 'mci conversion to ad', 'Alzheimer disease']}
Success in extracting columns from paper text from 28560309-PMC5440281
Successfully retrieve list of tables' id
Success in extracting from 27899424-PMC5237405 with table_link_to_excel


/Users/justpqa/advpai/gwas_formatting_engine.py:649: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_ref_col[col] = df[col]
/Users/justpqa/advpai/gwas_formatting_engine.py:649: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_ref_col[col] = df[col]
/Users/justpqa/advpai/gwas_formatting_engine.py:598: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pand

Success in harmonizing from 27899424-PMC5237405
Success in converting to number from 27899424-PMC5237405
27899424 PMC5237405
{'Population': [], 'Cohort': ['ADGC', 'IGAP', 'IPDGC', 'IFGC'], 'Stage': ['GLOBAL: discovery', 'GLOBAL: replication', 'IFGC Phase I', 'IFGC Phase II', 'Meta-analysis'], 'Imputation': ['GLOBAL: imputed to 1000 Genomes'], 'Study type': ['GLOBAL: genome-wide association studies (GWAS)'], 'Phenotype': ['FTD', 'AD', 'PD', 'Alzheimer’s disease', 'immune response', 'endolysosomal processes', 'intracellular vesicular trafficking', 'DNA/chromatin-associated metabolism', 'neurodegenerative disease', 'genetic traits for immune-mediated diseases', 'genetic variation in the HLA region', 'genetic pleiotropy', 'complex traits', 'genetic overlap', 'SNPs']}
Success in extracting columns from paper text from 27899424-PMC5237405


In [39]:
# matching_dict = {
#     "population": {
#         "confirmed": {
#             "European": ["European"],
#             "European ancestry": ["European ancestry"],
#             "European (EUR)": ["European"],
#             "East Asian": ["East Asian"],
#         },
#         "component_matches": {
#             "European ancestry + African American": [
#                 "European ancestry",
#             ],
#             "East Asian + European ancestry + African American": [
#                 "East Asian",
#                 "European ancestry",
#                 "East Asian, European ancestry",
#             ],
#         },
#         "non_matches": {
#             "": ["NHW", "European", "Meta-analysis"],
#             "European (EUR) + African American (AA)": ["NHW"],
#             "European (EUR)": [
#                 "NHW",
#                 "Non-Hispanic White",
#                 "European ancestry",
#                 "white",
#             ],
#             "European": ["European ancestry"],
#             "Chinese": ["White, black, other", "NHW"],
#             "African American": ["African ancestry"],
#         },
#     },

#     "cohort": {
#         "confirmed": {
#             "IFGC phase i": ["IFGC"],
#             "fhs": ["FHS"],
#             "chs": ["CHS"],
#             "hrs": ["HRS"],
#             "loadfs": ["NIA-LOAD"],
#             "ADNI": ["ADNI"],
#             "ADGC": ["ADGC"],
#             "IGAP": ["IGAP"],
#             "UK Biobank": ["UKBB"],
#         },
#         "component_matches": {
#             "loadfs + fhs + chs + hrs": [
#                 "CHS",
#                 "FHS",
#                 "HRS",
#                 "NIA-LOAD",
#             ],
#             "ADNI + ADGC": [
#                 "ADNI",
#                 "ADGC",
#             ],
#             "ADNI-1 + ADNI + ADNI-GO + ADNI-2": [
#                 "ADNI",
#             ],
#             "CHARGE + ADGC + IGAP + UK Biobank + EADI": [
#                 "CHARGE",
#                 "ADGC",
#                 "EADI",
#             ],
#             "ADSP + ADGC": [
#                 "ADGC",
#             ],
#         },
#         "non_matches": {
#             "IFGC phase i": ["UKBEC"],
#             "": ["IGAP", "MAGIC"],
#             "UK Biobank": [
#                 "HRS",
#                 "CHS",
#                 "LOADFS",
#                 "FHS",
#                 "ADSP, IGAP, PGC-ALZ",
#                 "ADSP, IGAP, PGC-ALZ, UKBB",
#             ],
#             "ADNI": [
#                 "ADNI, BIOCARD, Knight-ADRC, MAYO, SWEDEN, UPENN, UW",
#                 "ADNI, Knight-ADRC",
#                 "IGAP",
#                 "ROS/MAP",
#             ],
#             "ADNI + ADGC": [
#                 "East Asian Cohort (Korean National Research Center for Dementia), Japanese (Japanese Genetics Study Consortium of AlzheimerÕs Disease), Korean ( Korean Genome and Epidemiology Study)"
#             ],
#             "IGAP": ["PGC2-BIP"],
#             "CHARGE + ADNI + ADGC + EADI": [
#                 "GAME-ON",
#                 "IGAP, GAME-ON",
#                 "IGAP",
#             ],
#             "CHARGE + ADGC + IGAP + UK Biobank + EADI": [
#                 "ADGC, CHARGE, EADI, GERAD",
#             ],
#             "Mayo Clinic Brain eQTL": ["ADGC"],
#             "ADSP + ADGC": ["ADNI", "UKBEC", "NABEC"],
#             "UK Biobank": ["UKBB"],  # confirmed exact normalization exists; keep this if you want strict exact-match override logic outside this dict
#             "ADNI": ["IGAP"],        # repeated intentionally in non-match set context
#             "ADSP + ADGC": ["ADNI", "UKBEC", "NABEC"],
#         },
#     },

#     "stage": {
#         "confirmed": {
#             "Meta-analysis": ["Meta-analysis"],
#             "Discovery": ["Discovery"],
#             "Replication": ["Replication/Validation"],
#             "": [""],
#         },
#         "component_matches": {
#             "Meta-analysis + Discovery + Replication": [
#                 "Meta-analysis",
#                 "Discovery",
#                 "Replication/Validation",
#             ],
#             "Meta-analysis + Replication": [
#                 "Meta-analysis",
#             ],
#             "Stage 2 + Stage 1 + Meta-analysis": [
#                 "Meta-analysis",
#             ],
#         },
#         "non_matches": {
#             "Meta-analysis": ["Discovery", "Replication/Validation"],
#             "": ["Discovery", "Meta-analysis", "Joint-analysis"],
#             "Meta-analysis + Replication": ["Stage 2"],
#             "Meta-analysis + Discovery + Replication": [
#                 "Discovery, Stage 2, Stage 3A",
#                 "Stage 3B (Stage 2 + Stage 3A)",
#                 "Stage 1, Stage 3B (Stage 2 + Stage 3A)",
#             ],
#         },
#     },

#     "imputation": {
#         "confirmed": {
#             "1000 genome": ["1000G"],
#             "1000 Genomes (1000G)": ["1000G"],
#             "1000 Genomes": ["1000G"],
#             "Haplotype Reference Consortium (HRC)": ["HRC"],
#             "1000 genome phase 3 v5 reference panel": ["1000G Phase 3 v5"],
#         },
#         "component_matches": {},
#         "non_matches": {
#             "1000 Genomes (1000G)": ["1000G Phase 3 (release Oct 2014)"],
#             "1000 Genomes": ["1000G Phase 3"],
#             "1000 genome": ["HapMap2"],
#             "": ["1000G (release Mar 2012)", "Imputed"],
#             "Haplotype Reference Consortium (HRC) + 1000 Genomes (1000G)": [
#                 "HRC r1.1",
#                 "1000G Phase 3 (release Oct 2014)",
#                 "HRC r1.1, 1000G Phase 3",
#             ],
#             "eagle v2.3 phasing + 1000 genome phase 3 v5 reference panel": [
#                 "Imputed",
#             ],
#         },
#     },

#     "study type": {
#         "confirmed": {
#             "GWAS": ["SNP-based"],
#             "genome-wide association study": ["SNP-based"],
#             "Whole Genome Association Study": ["SNP-based"],
#             "Genotyping": ["SNP-based"],
#             "Sequencing": ["SNP-based"],
#         },
#         "component_matches": {
#             "Whole Genome Association Study + Genotyping": ["SNP-based"],
#             "Genotyping + Sequencing": ["SNP-based"],
#         },
#         "non_matches": {
#             "": ["SNP-based", "Gene-based"],
#             "GWAS": ["Gene-based"],
#             "Genotyping + Sequencing": ["Interaction(SNP)"],
#             "Genotyping + Whole Genome Sequencing (WGS)": ["Interaction(SNP)"],
#             "Whole Exome Sequencing (WES)": ["SNP-based"],
#         },
#     },

#     "phenotype": {
#         "confirmed": {
#             "frontotemporal dementia": ["Frontotemporal Dementia"],
#             "Alzheimer’s disease": ["AD"],
#             "Dementia + Alzheimer’s disease": ["AD"],
#             "Alzheimer’s disease + dementia": ["AD"],
#             "Fasting Glucose": ["Fasting Glucose"],
#             "np": ["Neuritic plaque (NP)"],
#             "nft": ["Neurofibrillary tangles (NFT)"],
#             "caa": ["Cerebral amyloid angiopathy (CAA)"],
#             "Cerebrospinal fluid phosphorylated tau ptau181": ["CSF P-tau181p"],
#             "Cerebrospinal fluid amyloid-beta142": ["CSF Ab1-42"],
#             "Coronary artery disease": ["Coronary artery disease (CAD)"],
#             "Type 2 diabetes": ["Type 2 diabetes (T2D)"],
#             "Low-density lipoprotein": ["Low-density lipoprotein (LDL)"],
#             "Waist hip ratio": ["Waist-to-hip ratio (WHR)"],
#             "Triglycerides": ["Total triglycerides (TG)"],
#             "Total cholesterol": ["Total cholesterol (TC)"],
#             "High-density lipoprotein": ["High-density lipoprotein (HDL)"],
#             "BMI": ["Body-mass index (BMI)"],
#         },
#         "component_matches": {
#             "frontotemporal dementia + Alzheimer’s disease + Parkinson's disease": [
#                 "Frontotemporal Dementia",
#             ],
#             "Blood pressure + Alzheimer’s disease + LDL cholesterol + Type 2 diabetes + BMI": [
#                 "AD",
#             ],
#             "Blood pressure + Alzheimer’s disease": [
#                 "AD",
#             ],
#             "Alzheimer’s disease + Type 2 diabetes": [
#                 "AD",
#             ],
#             "cancer + Alzheimer’s disease": [
#                 "AD",
#             ],
#             "np + nft + caa + Alzheimer’s disease": [
#                 "Neuritic plaque (NP)",
#                 "Neurofibrillary tangles (NFT)",
#                 "Cerebral amyloid angiopathy (CAA)",
#                 "Neuritic plaque (NP)+Neurofibrillary tangles (NFT)",
#                 "Neuritic plaque (NP)+Cerebral amyloid angiopathy (CAA)",
#                 "Neurofibrillary tangles (NFT)+Cerebral amyloid angiopathy (CAA)",
#                 "AD",
#             ],
#             "educational attainment + Alzheimer’s disease + cognitive ability": [
#                 "AD",
#             ],
#             "Cerebrospinal fluid clusterin + Cerebrospinal fluid phosphorylated tau ptau181 + Alzheimer’s disease + Cerebrospinal fluid amyloid-beta142": [
#                 "CSF P-tau181p",
#                 "CSF Ab1-42",
#             ],
#             "Coronary artery disease + Type 2 diabetes + Low-density lipoprotein + Waist hip ratio + Triglycerides + Total cholesterol + High-density lipoprotein + BMI": [
#                 "Coronary artery disease (CAD)",
#                 "Type 2 diabetes (T2D)",
#                 "Low-density lipoprotein (LDL)",
#                 "Waist-to-hip ratio (WHR)",
#                 "Total triglycerides (TG)",
#                 "Total cholesterol (TC)",
#                 "High-density lipoprotein (HDL)",
#                 "Body-mass index (BMI)",
#             ],
#             "MCI conversion to AD + Alzheimer’s disease": [
#                 "AD",
#             ],
#             "Blood pressure + Alzheimer’s disease": [
#                 "AD",
#             ],
#             "Alzheimer’s disease + dementia": [
#                 "AD",
#             ],
#         },
#         "non_matches": {
#             "Blood pressure + Alzheimer’s disease + LDL cholesterol + Type 2 diabetes + BMI": [
#                 "Fasting Insulin",
#                 "ABCB9 (ILMN_2343047) expression",
#                 "ATG10 expression in Nucleus accumbens",
#                 "IRAK3 (ILMN_1913678) expression in peripheral blood",
#                 "AHSA2 expression in Cerebellum",
#                 "CRHR1 expression in Medulla",
#                 "LRRC37A4 expression in Cerebellum",
#                 "KANSL1 expression in Hippocampus",
#                 "LRRC37A2 expression in Temporal cortex",
#                 "HLA-DPA1 expression in Frontal cortex",
#             ],
#             "Blood pressure + Alzheimer’s disease": [
#                 "MYBPC3 (11725151_at) expression in blood",
#                 "AK9 (t2969159) expression in frontal cortex",
#                 "AGPAT1 (11751668_a_at) expression in blood",
#                 "TRIM4 (11736388_a_at) expression in blood",
#                 "PILRB (11730022_a_at) expression in blood",
#                 "PTK2B (11720981_a_at) expression in blood",
#                 "IER2 (t3822216) expression in temporal cortex",
#                 "EID2B (t3862068) expression in frontal cortex",
#                 "BIN1 (11719631_s_at) expression in blood",
#                 "DLGAP1 (ILMN_2380779) expression in frontal cortex",
#                 "NETO1 (ILMN_1783168) expression in frontal cortex",
#                 "MS4A6A (11716846_a_at) expression in blood",
#                 "MS4A4A (11751570_a_at) expression in blood",
#             ],
#             "Alzheimer’s disease": [
#                 "CSF T-tau",
#                 "CSF Ab1-42",
#                 "Logic memory delayed (LMdT) recall test",
#                 "Logical memory immediate (LMiT) recall test",
#                 "Hippocampal volume (HPV)",
#                 "CSF P-tau181p",
#                 "Amyloid Burden",
#                 "Neuronal Neurofibrillary Tangles",
#                 "Cortical atrophy in Medial temporal cortex",
#                 "Cortical atrophy in Hippocampal Volume",
#                 "Cortical atrophy in Precuneus",
#                 "AD risk",
#                 "Age at onset",
#                 "AD Progression (based on CDR Score)",
#             ],
#             "Dementia + Alzheimer’s disease": [
#                 "Bipolar disorder (BIP)",
#             ],
#             "Alzheimer’s disease + Type 2 diabetes": [
#                 "Age at onset",
#             ],
#             "cancer + Alzheimer’s disease": [
#                 "Breast Cancer",
#                 "Lung Cancer",
#                 "AD + Breast cancer + Lung Cancer",
#             ],
#             "MCI conversion to AD + Alzheimer’s disease": [
#                 "ADAS-Cog scores (Longitudinal Alzheimer's Disease Assessment Scale-Cognition scores)",
#             ],
#             "Coronary artery disease + Type 2 diabetes + Low-density lipoprotein + Waist hip ratio + Triglycerides + Total cholesterol + High-density lipoprotein + BMI": [
#                 "AD",
#             ],
#             "educational attainment + Alzheimer’s disease + cognitive ability": [
#                 "AD-by-proxy",
#                 "AD + AD-by-proxy",
#             ],
#         },
#     },
# }

# import re
# import json
# from pathlib import Path
# from collections import Counter, defaultdict
# from typing import Dict, List, Tuple, Any

# import pandas as pd


# TARGET_COLUMNS = [
#     "Population",
#     "Cohort",
#     "Stage",
#     "Imputation",
#     "Study type",
#     "Phenotype",
# ]

# SNP_COLUMN = "SNP"


# def normalize_text(x: Any) -> str:
#     if pd.isna(x):
#         return ""
#     s = str(x).strip()
#     if s.lower() in {"nan", "none", "null"}:
#         return ""
#     s = s.replace("’", "'").replace("“", '"').replace("”", '"')
#     s = re.sub(r"\s+", " ", s)
#     return s


# def split_plus_terms(s: str) -> List[str]:
#     s = normalize_text(s)
#     if not s:
#         return []
#     return [p.strip() for p in s.split("+") if p.strip()]


# def validate_required_columns(df: pd.DataFrame, df_name: str) -> None:
#     required = TARGET_COLUMNS + [SNP_COLUMN]
#     missing = [c for c in required if c not in df.columns]
#     if missing:
#         raise ValueError(
#             f"{df_name} is missing required columns: {missing}. "
#             f"Expected columns: {required}"
#         )


# def normalize_mapping_dict(raw_mapping_dict: Dict[str, Any]) -> Dict[str, Any]:
#     out = {}
#     for col, col_dict in raw_mapping_dict.items():
#         out[col] = {
#             "confirmed": {},
#             "component_matches": {},
#             "non_matches": {},
#         }
#         for section in ["confirmed", "component_matches", "non_matches"]:
#             for k, vals in col_dict.get(section, {}).items():
#                 nk = normalize_text(k)
#                 uniq = []
#                 seen = set()
#                 for v in vals:
#                     nv = normalize_text(v)
#                     if nv not in seen:
#                         uniq.append(nv)
#                         seen.add(nv)
#                 out[col][section][nk] = uniq
#     return out


# def build_reverse_confirmed_map(col_dict: Dict[str, Any]) -> Dict[str, List[str]]:
#     reverse = {}
#     for pred_term, gt_terms in col_dict.get("confirmed", {}).items():
#         for gt in gt_terms:
#             reverse.setdefault(gt, []).append(pred_term)
#     return reverse


# def is_known_nonmatch(mapping_dict: Dict[str, Any], column: str, pred_val: str, gt_val: str) -> bool:
#     pred_val = normalize_text(pred_val)
#     gt_val = normalize_text(gt_val)
#     return gt_val in mapping_dict.get(column, {}).get("non_matches", {}).get(pred_val, [])


# def pred_value_covers_gt_value(
#     mapping_dict: Dict[str, Any],
#     column: str,
#     pred_val: str,
#     gt_val: str,
# ) -> bool:
#     pred_val = normalize_text(pred_val)
#     gt_val = normalize_text(gt_val)

#     if gt_val == "":
#         return True
#     if pred_val == "":
#         return False

#     if is_known_nonmatch(mapping_dict, column, pred_val, gt_val):
#         return False

#     col_dict = mapping_dict.get(column, {})
#     confirmed = col_dict.get("confirmed", {})
#     component_matches = col_dict.get("component_matches", {})
#     reverse_confirmed = build_reverse_confirmed_map(col_dict)

#     pred_parts = split_plus_terms(pred_val)
#     gt_parts = split_plus_terms(gt_val) or [gt_val]

#     # Ground truth with "+" means all GT components must be covered by the same pred value.
#     for gt_component in gt_parts:
#         covered = False

#         if pred_val == gt_component:
#             covered = True
#         elif gt_component in pred_parts:
#             covered = True
#         elif gt_component in confirmed.get(pred_val, []):
#             covered = True
#         elif gt_component in component_matches.get(pred_val, []):
#             covered = True
#         else:
#             for part in pred_parts:
#                 if gt_component in confirmed.get(part, []):
#                     covered = True
#                     break
#                 if gt_component in component_matches.get(part, []):
#                     covered = True
#                     break

#         if not covered:
#             for candidate_pred in reverse_confirmed.get(gt_component, []):
#                 if pred_val == candidate_pred or candidate_pred in pred_parts:
#                     covered = True
#                     break

#         if not covered:
#             return False

#     return True


# def collect_values_by_snp(df: pd.DataFrame, column: str) -> Dict[str, List[str]]:
#     by_snp = defaultdict(list)
#     for _, row in df.iterrows():
#         snp = normalize_text(row[SNP_COLUMN])
#         val = normalize_text(row[column])
#         by_snp[snp].append(val)
#     return dict(by_snp)


# def multiset_cover_gt_with_pred(
#     mapping_dict: Dict[str, Any],
#     column: str,
#     gt_values: List[str],
#     pred_values: List[str],
# ) -> Tuple[bool, Dict[str, Any]]:
#     """
#     GT and Pred are treated as multisets.
#     Each predicted row/value can cover at most one GT value occurrence.
#     """
#     gt_values = [normalize_text(x) for x in gt_values]
#     pred_values = [normalize_text(x) for x in pred_values]

#     # Blank GT values are automatically covered
#     gt_nonblank = [x for x in gt_values if x != ""]
#     gt_blank_count = len(gt_values) - len(gt_nonblank)

#     # Build bipartite-style greedy matching:
#     # For each GT item, find one unused Pred item that can cover it.
#     used_pred = [False] * len(pred_values)
#     matched_pairs = []
#     uncovered_gt = []

#     # Sort GT by rarity/difficulty: exact blanks removed, now try longer/specific strings first
#     gt_order = sorted(
#         enumerate(gt_nonblank),
#         key=lambda x: (-len(x[1]), x[1])
#     )

#     for gt_idx, gt_val in gt_order:
#         found = False
#         for pred_idx, pred_val in enumerate(pred_values):
#             if used_pred[pred_idx]:
#                 continue
#             if pred_value_covers_gt_value(mapping_dict, column, pred_val, gt_val):
#                 used_pred[pred_idx] = True
#                 matched_pairs.append((gt_val, pred_val))
#                 found = True
#                 break
#         if not found:
#             uncovered_gt.append(gt_val)

#     covered = len(uncovered_gt) == 0

#     return covered, {
#         "gt_total": len(gt_values),
#         "gt_blank_count": gt_blank_count,
#         "gt_nonblank_count": len(gt_nonblank),
#         "pred_total": len(pred_values),
#         "matched_pairs": matched_pairs,
#         "uncovered_gt": uncovered_gt,
#         "gt_counter": dict(Counter(gt_values)),
#         "pred_counter": dict(Counter(pred_values)),
#     }


# def evaluate_tables_snp_multiset(
#     gt_path: str,
#     pred_path: str,
#     raw_mapping_dict: Dict[str, Any],
#     output_prefix: str = None,
# ) -> Tuple[pd.DataFrame, Dict[str, Any]]:
#     mapping_dict = normalize_mapping_dict(raw_mapping_dict)

#     gt_df = pd.read_csv(gt_path)
#     pred_df = pd.read_csv(pred_path)

#     validate_required_columns(gt_df, "Ground truth table")
#     validate_required_columns(pred_df, "Predicted table")

#     gt_df = gt_df.copy()
#     pred_df = pred_df.copy()

#     gt_df[SNP_COLUMN] = gt_df[SNP_COLUMN].map(normalize_text)
#     pred_df[SNP_COLUMN] = pred_df[SNP_COLUMN].map(normalize_text)

#     gt_snps = list(gt_df[SNP_COLUMN].unique())
#     pred_snp_set = set(pred_df[SNP_COLUMN].unique())

#     records = []
#     column_covered_counts = {col: 0 for col in TARGET_COLUMNS}

#     for snp in gt_snps:
#         gt_snp_rows = gt_df[gt_df[SNP_COLUMN] == snp]
#         pred_snp_rows = pred_df[pred_df[SNP_COLUMN] == snp] if snp != "" else pred_df

#         row = {
#             "SNP": snp,
#             "gt_row_count": len(gt_snp_rows),
#             "pred_row_count": len(pred_snp_rows),
#             "snp_present_in_pred": (snp in pred_snp_set) if snp != "" else True,
#         }

#         for col in TARGET_COLUMNS:
#             gt_values = gt_snp_rows[col].map(normalize_text).tolist()
#             pred_values = pred_snp_rows[col].map(normalize_text).tolist()

#             covered, details = multiset_cover_gt_with_pred(
#                 mapping_dict=mapping_dict,
#                 column=col,
#                 gt_values=gt_values,
#                 pred_values=pred_values,
#             )

#             row[f"{col}__covered"] = covered
#             row[f"{col}__gt_values"] = json.dumps(details["gt_counter"], ensure_ascii=False)
#             row[f"{col}__pred_values"] = json.dumps(details["pred_counter"], ensure_ascii=False)
#             row[f"{col}__uncovered_gt"] = json.dumps(details["uncovered_gt"], ensure_ascii=False)
#             row[f"{col}__matched_pairs"] = json.dumps(details["matched_pairs"], ensure_ascii=False)

#             if covered:
#                 column_covered_counts[col] += 1

#         records.append(row)

#     results_df = pd.DataFrame(records)

#     summary = {
#         "gt_unique_snps": len(gt_snps),
#         "pred_unique_snps": pred_df[SNP_COLUMN].nunique(),
#         "column_coverage_by_snp": {
#             col: {
#                 "covered_snps": int(column_covered_counts[col]),
#                 "total_snps": int(len(gt_snps)),
#                 "fraction": float(column_covered_counts[col] / len(gt_snps)) if gt_snps else 0.0,
#             }
#             for col in TARGET_COLUMNS
#         }
#     }

#     if output_prefix:
#         Path(output_prefix).parent.mkdir(parents=True, exist_ok=True)
#         results_df.to_csv(f"{output_prefix}_snp_multiset_results.csv", index=False)
#         with open(f"{output_prefix}_snp_multiset_summary.json", "w", encoding="utf-8") as f:
#             json.dump(summary, f, indent=2, ensure_ascii=False)

#     return results_df, summary

# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]

# for pmid, pmcid in test_papers_info:
#     print(pmid, pmcid)
#     gt_path = f"test_tables/{pmid}_{pmcid}.csv"
#     pred_path = f"pred_tables/{pmid}_{pmcid}.csv"

#     results_df, summary = evaluate_tables_snp_multiset(
#         gt_path=gt_path,
#         pred_path=pred_path,
#         raw_mapping_dict=matching_dict,
#         output_prefix="new_eval"
#     )

#     print(summary)

In [40]:
# referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
# gwas_information_retriever = GWASInformationRetriever(referencing_col_require_rag_df, use_hf = True, device = "mps")
# col_require_rag_to_possible_info = gwas_information_retriever.extract_possible_info_from_paper(30413934, "PMC6358498")

In [41]:
# referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
# gwas_formatting_engine = GWASFormattingEngine(referencing_col_df)

# referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
# gwas_information_retriever = GWASInformationRetriever(referencing_col_require_rag_df, use_hf = True, device = "mps")

# pmid, pmcid = 28247064, "PMC5613285"
# harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
# for file_name in os.listdir("tables"):
#     if str(pmid) in file_name and pmcid in file_name:
#         if ".xlsx" in file_name:
#             df = pd.read_excel(f"tables/{file_name}")
#         else:
#             df = pd.read_csv(f"tables/{file_name}")

#         # save matching dict for debug
#         # file_name_to_matching[file_name] = gwas_formatting_engine.match_many_col_to_ref_col(df)

#         harmonized_df = gwas_formatting_engine.format_original_table(df, remove_unique_col = True)
#         if harmonized_df_all is None:
#             harmonized_df_all = harmonized_df.copy()
#         else:
#             harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
# harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)

# harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
# harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
# int_col = ["Chr"]
# for c in int_col:
#     harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
# numerical_col_with_many_numbers = ["Effect"]
# for c in numerical_col_with_many_numbers:
#     harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
# float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
# for c in float_col:
#     harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
#     if float_col[c] is not None:
#         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
# harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)

# col_require_rag_to_possible_info = gwas_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
# harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
# # if we have cols that is notes => use info in that to check for possible mapping
# notes_col = [col for col in harmonized_df_all.columns if "notes" in col]
# if len(notes_col) == 0:
#     for col in col_require_rag_to_possible_info:
#         if len(col_require_rag_to_possible_info[col]) == 0:
#             harmonized_df_all[col] = pd.NA
#         else:
#             harmonized_df_all[col] = combine_possible_info(col_require_rag_to_possible_info[col])
# else:
#     # use info in notes to map, for each info, find the best one
#     for col in col_require_rag_to_possible_info:
#         if len(col_require_rag_to_possible_info[col]) == 0:
#             harmonized_df_all[col] = pd.NA
#         else:
#             used_col = []
#             for n_col in notes_col:
#                 unique_value = harmonized_df_all[[n_col]].dropna()[n_col].unique().tolist()
#                 similarity_score = calculate_similarity_scores(col_require_rag_to_possible_info[col], unique_value) # #possible info * #unique value
#                 best_inx = torch.argmax(similarity_score, dim = 0)
#                 unique_value_to_possible_info = {}
#                 for i, u in enumerate(unique_value):
#                     unique_value_to_possible_info[u] = col_require_rag_to_possible_info[col][best_inx[i]]
#                 harmonized_df_all[f"{col} from {n_col}"] = harmonized_df_all[n_col].apply(lambda x: unique_value_to_possible_info.get(x, ""))
#                 used_col.append(f"{col} from {n_col}")
#             harmonized_df_all[col] = harmonized_df_all[used_col].apply(lambda x: combine_possible_info(x), axis = 1)
#             harmonized_df_all[col] = harmonized_df_all[col].apply(lambda x: pd.NA if len(x) == 0 else x)
#             harmonized_df_all = harmonized_df_all.drop(used_col, axis = 1)
# harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)

In [42]:
for f in file_name_to_matching:
    for ref_col in file_name_to_matching[f]:
        for i in range(len(file_name_to_matching[f][ref_col])):
            file_name_to_matching[f][ref_col][i] = (file_name_to_matching[f][ref_col][i][0], float(file_name_to_matching[f][ref_col][i][1]))
with open("test_matching_dict.json", "w") as f:
    json.dump(file_name_to_matching, f, indent=4)

In [43]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]

# id_to_table_id_lst = {}
# for pmid, pmcid in test_papers_info:
#     id_to_table_id_lst[pmcid] = {
#         "PMID": pmid,
#         "table_id_lst": extract_table_id_lst_from_pmc(pmcid)
#     }

# with open("id_to_table_id_lst.json", "w") as f:
#     json.dump(id_to_table_id_lst, f, indent=4)

In [44]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]

# df = pd.read_csv("test_tables/ADVP_1026_v3p8_extracted.txt", sep = "\t", encoding="cp1252")
# df = df.replace("NR", pd.NA)
# df = df.rename({"Cohort_simplified_no_counts": "Cohort"}, axis = 1)
# pmid_pmcid_to_special_columns_val = {}

# for pmid, pmcid in test_papers_info:
#     pmid_pmcid_to_special_columns_val[f"{pmid}-{pmcid}"] = {}
#     temp_df = df[(df["Pubmed ID"] == pmid) & (df["PMCID"] == pmcid)].reset_index().drop("index", axis = 1)
#     for col in ["Population", "Cohort", "Stage"]:
#         pmid_pmcid_to_special_columns_val[f"{pmid}-{pmcid}"][col] = temp_df[[col]].dropna()[col].unique().tolist()

# with open("pmid_pmcid_to_special_columns_val.json", "w") as f:
#     json.dump(pmid_pmcid_to_special_columns_val, f, indent=4) 

In [45]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]

# referencing_col_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
# gwas_information_retriever = GWASInformationRetriever(referencing_col_df, device = "mps")

# pmid_pmcid_to_special_columns_val = {}
# for pmid, pmcid in tqdm(test_papers_info):
#     pmid_pmcid_to_special_columns_val[f"{pmid}-{pmcid}"] = gwas_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
# with open("pmid_pmcid_to_special_columns_val_pred.json", "w") as f:
#     json.dump(pmid_pmcid_to_special_columns_val, f, indent=4) 

In [46]:
# # eval script
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"),  (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]

# with open("pmid_pmcid_to_special_columns_val.json", "r") as f:
#     pmid_pmcid_to_special_columns_val = json.load(f) 

# with open("pmid_pmcid_to_special_columns_val_pred.json", "r") as f:
#     pmid_pmcid_to_special_columns_val_pred = json.load(f) 

# embeddings_model = AutoModel.from_pretrained("NeuML/pubmedbert-base-embeddings")
# embeddings_model_tokenizer = AutoTokenizer.from_pretrained("NeuML/pubmedbert-base-embeddings")

# def cosine_similarity(s1, s2):
#     input = embeddings_model_tokenizer([s1, s2], padding=True, truncation=True, return_tensors='pt')

#     # get token embeddings
#     with torch.no_grad():
#         output = embeddings_model(**input)
#     token_embeddings = output[0]

#     # extract mask and mean pooling for sentence embeddings
#     input_mask_expanded = input['attention_mask'].unsqueeze(-1).expand(token_embeddings.size()).float()
#     embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

#     # final normalization
#     embeddings = F.normalize(embeddings, p=2, dim=1)

#     return torch.sum(embeddings[0, :] * embeddings[1, :])

# score_by_paper = {}
# mean_score = {}
# for pmid, pmcid in test_papers_info:
#     id = f"{pmid}-{pmcid}"
#     score_by_paper[id] = {}
#     for col in ["Population", "Cohort", "Stage"]:
#         s1, s2 = pmid_pmcid_to_special_columns_val[id][col], pmid_pmcid_to_special_columns_val_pred[id][col]
#         s2 = [str(item) for item in s2]
#         s1, s2 = " ".join(s1), " ".join(s2)
#         score_by_paper[id][col] = cosine_similarity(s1, s2)
#         mean_score[col] = mean_score.get(col, 0) + score_by_paper[id][col]
# for col in ["Population", "Cohort", "Stage"]:
#     mean_score[col] /= len(test_papers_info)
# print(mean_score)
# # test on llm, only top one are shown
# # qwen2.5-0.5b-it {'Population': tensor(0.4555), 'Cohort': tensor(0.3733), 'Stage': tensor(0.2460)}
# # qwen2.5-1.5b-it {'Population': tensor(0.5661), 'Cohort': tensor(0.5251), 'Stage': tensor(0.2988)}
# # qwen2.5-3b-it 8bit {'Population': tensor(0.5559), 'Cohort': tensor(0.4690), 'Stage': tensor(0.6046)}
# # qwen2.5-7b-it 4bit {'Population': tensor(0.5466), 'Cohort': tensor(0.5456), 'Stage': tensor(0.5054)}

In [47]:
# score_by_paper